# Stage 2: cleaning

This notebook turns the raw OpenF1 files in `data/raw/` into clean tables in `data/processed/`.

- **Part 1 (sections 1 to 9): one lap table and one pit-stop table.**
- **Part 2 (sections 10 to 17): the race situation of every lap** (track status, restart laps, weather, wet races), the pit loss of every race, and the flags `clean_lap` and `strategy_stop` that later stages filter on.

Principle: *flag, don't delete*. Every lap is kept and gets flags, because later stages need different subsets (the tyre model needs clean laps, the overtaking model needs every lap). Only rows that are genuinely broken are removed, and they are counted in the cleaning table.

Input: `data/raw/` (from `01_get_data.py`). Output: `data/processed/laps.parquet`, `data/processed/pit_stops.parquet`, `data/processed/races.parquet`, `results/cleaning_table.csv`, `results/stint_change_exceptions.csv`.


In [1]:
import time

import numpy as np
import pandas as pd

from helpers import DATA_PROCESSED, RESULTS, load_races, load_raw

start_time = time.time()

# One car in one race.
KEY = ["session_key", "driver_number"]
LAP_KEY = KEY + ["lap_number"]

# A lap end time (next lap's start) that differs from start + lap_duration
# by more than this is flagged. Timing is to the millisecond, so 0.5 s is
# not a rounding difference.
TIME_MISMATCH_SECONDS = 0.5

# Part 2. A race is wet if any car used one of these compounds.
WET_COMPOUNDS = ["INTERMEDIATE", "WET"]
# Pit loss: the driver's typical lap is the median of up to this many normal
# green laps on each side of the stop.
TYPICAL_LAP_WINDOW = 3
# A race with fewer usable stops than this takes the median of its circuit.
MIN_STOPS_FOR_PIT_LOSS = 3
# A stop whose pit-lane time is more than this many seconds above the median
# pit-lane time of its race (a slow stop or a served penalty) is left out of
# the pit loss estimate.
SLOW_STOP_LANE_TIME = 5
# The same for the six 2023 races without pit data, measured on the lap-based
# pit loss against the median pit loss of the race.
SLOW_STOP_PIT_LOSS = 8
# A stop that loses more than this many seconds above the pit loss of its race
# is not a strategy stop (likely repairs). Penalties of 5 or 10 s stay below it.
SLOW_STOP_FOR_STRATEGY = 20
# A lap slower than this multiple of the driver's median lap in the same stint
# is an outlier for the pace model.
OUTLIER_RATIO = 1.07

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 200)

# The cleaning table: one row per step.
cleaning_steps = []


def add_step(step, laps_before, laps_affected, action, reason):
    """Add one row to the cleaning table."""
    cleaning_steps.append(
        {"step": step, "laps_before": int(laps_before), "laps_affected": int(laps_affected),
         "action": action, "reason": reason}
    )

## 1. Load the raw tables

The raw files keep the values as delivered, so column types differ between races. `load_raw` (in `helpers.py`) converts the listed columns explicitly while combining the races.

In [2]:
races = load_races()

laps_raw = load_raw(
    "laps",
    columns=["session_key", "driver_number", "lap_number", "date_start", "lap_duration",
             "duration_sector_1", "duration_sector_2", "duration_sector_3",
             "i1_speed", "i2_speed", "st_speed", "is_pit_out_lap"],
    number_columns=["lap_duration", "duration_sector_1", "duration_sector_2", "duration_sector_3",
                    "i1_speed", "i2_speed", "st_speed"],
    date_columns=["date_start"],
)
drivers = load_raw("drivers", columns=["session_key", "driver_number", "name_acronym", "team_name"])
results = load_raw(
    "session_result",
    columns=["session_key", "driver_number", "position", "number_of_laps", "dnf", "dns", "dsq"],
    number_columns=["position", "number_of_laps"],
)
stints_raw = load_raw("stints", columns=["session_key", "driver_number", "stint_number", "lap_start",
                                         "lap_end", "compound", "tyre_age_at_start"])
pit_raw = load_raw(
    "pit",
    columns=["session_key", "driver_number", "lap_number", "date", "pit_duration", "lane_duration",
             "stop_duration"],
    number_columns=["pit_duration", "lane_duration", "stop_duration"],
    date_columns=["date"],
)
race_control = load_raw("race_control", columns=["session_key", "date", "category", "message"],
                        date_columns=["date"])

weather = load_raw(
    "weather",
    columns=["session_key", "date", "track_temperature", "air_temperature", "rainfall"],
    number_columns=["track_temperature", "air_temperature", "rainfall"],
    date_columns=["date"],
)

print(f"races: {len(races)}")
for name, table in [("laps", laps_raw), ("drivers", drivers), ("results", results),
                    ("stints", stints_raw), ("pit", pit_raw), ("race_control", race_control),
                    ("weather", weather)]:
    print(f"{name}: {len(table):,} rows, {table['session_key'].nunique()} races")

races: 91
laps: 101,988 rows, 91 races
drivers: 1,859 rows, 91 races
results: 1,859 rows, 91 races
stints: 5,301 rows, 91 races
pit: 3,202 rows, 85 races
race_control: 10,164 rows, 91 races
weather: 14,600 rows, 91 races


## 2. One row per driver per lap

Year and circuit come from the race index, the driver code and team from the drivers table **of the same race** (four drivers changed team during a season). Circuits are identified by the circuit name, never by the country.

In [3]:
laps = laps_raw.merge(races[["session_key", "year", "circuit"]], on="session_key", how="left",
                      validate="many_to_one")
driver_names = drivers.rename(columns={"name_acronym": "driver", "team_name": "team"})
laps = laps.merge(driver_names, on=KEY, how="left", validate="many_to_one")
laps = laps.sort_values(LAP_KEY).reset_index(drop=True)

rows_raw = len(laps)
exact_duplicates = int(laps.duplicated().sum())
laps = laps.drop_duplicates().reset_index(drop=True)
add_step("combine the raw lap files of all races", rows_raw, 0, "none", "starting point")
add_step("exact duplicate rows", rows_raw, exact_duplicates, "removed", "the same row delivered twice")

assert not laps.duplicated(LAP_KEY).any(), "a driver has two rows for the same lap"
assert laps["driver"].notna().all() and laps["team"].notna().all(), "a lap has no driver code or team"
assert drivers.groupby(KEY)["team_name"].nunique().max() == 1, "a driver has two teams in one race"

print(f"lap rows: {len(laps):,}, exact duplicates removed: {exact_duplicates}")

lap rows: 101,988, exact duplicates removed: 0


Three races have 19 cars instead of 20. The missing car is the team-mate of the only driver whose team has one car in that race.

In [4]:
cars = drivers.groupby("session_key")["driver_number"].nunique()
normal_cars = drivers.merge(races[["session_key", "year"]], on="session_key").groupby("year")["session_key"].apply(
    lambda keys: cars[keys.unique()].max())
team_size = drivers.groupby(["session_key", "team_name"]).size()
one_car_teams = team_size[team_size == 1].reset_index()[["session_key", "team_name"]]
for row in one_car_teams.merge(races, on="session_key").sort_values("date").itertuples():
    # The missing car: a driver who drove for this team in another race of the season but not here.
    same_season = races.loc[races["year"] == row.year, "session_key"]
    team_drivers = drivers[drivers["session_key"].isin(same_season) & (drivers["team_name"] == row.team_name)]
    here = set(drivers.loc[drivers["session_key"] == row.session_key, "driver_number"])
    absent = team_drivers[~team_drivers["driver_number"].isin(here)]
    absent = absent.groupby(["driver_number", "name_acronym"]).size().sort_values(ascending=False)
    number, code = absent.index[0]
    print(f"{row.year} {row.circuit}: {cars[row.session_key]} cars, {row.team_name} has one car; "
          f"missing car {number} ({code})")

2023 Singapore: 19 cars, Aston Martin has one car; missing car 18 (STR)
2024 Melbourne: 19 cars, Williams has one car; missing car 2 (SAR)
2025 Catalunya: 19 cars, Aston Martin has one car; missing car 18 (STR)


## 3. Race distance, and rows after a driver's last lap

The race distance is the winner's number of laps in the results table. `classified_laps` is the driver's own number of completed laps there.

The lap file has rows beyond `classified_laps`:
- empty rows after the chequered flag for cars that finished: **removed** (no lap was driven);
- an empty lap row for a car that did not start: **removed**;
- the lap on which a car retired: **kept**, flagged `incomplete_lap`;
- any timed lap beyond `classified_laps` of a car that finished: **kept**, flagged `beyond_classified` (a lap after the chequered flag, never a racing lap).

In [5]:
winner_laps = results[results["position"] == 1].set_index("session_key")["number_of_laps"]
assert winner_laps.index.is_unique and len(winner_laps) == len(races), "not exactly one winner per race"

result_columns = results[KEY + ["number_of_laps", "dnf", "dns", "dsq"]].rename(
    columns={"number_of_laps": "classified_laps"})
laps = laps.merge(result_columns, on=KEY, how="left", validate="many_to_one")
assert laps["dnf"].notna().all(), "a driver with laps is not in the results"
laps["race_laps"] = laps["session_key"].map(winner_laps).astype(int)

time_columns = ["lap_duration", "duration_sector_1", "duration_sector_2", "duration_sector_3"]
no_times = laps[time_columns].isna().all(axis=1)
after_last_lap = laps["lap_number"] > laps["classified_laps"]

empty_after_flag = after_last_lap & no_times & ~laps["dnf"] & ~laps["dns"]
empty_non_starter = no_times & laps["dns"]
print("Rows after the driver's last classified lap:", int(after_last_lap.sum()))
print("  empty rows after the chequered flag (removed):", int(empty_after_flag.sum()))
print("  empty rows of cars that did not start (removed):", int(empty_non_starter.sum()))
print(laps.loc[empty_non_starter, ["year", "circuit", "driver", "lap_number", "classified_laps"]].to_string(index=False))

rows_before = len(laps)
add_step("empty rows after the chequered flag", rows_before, empty_after_flag.sum(), "removed",
         "row beyond the driver's classified laps with no lap or sector time; the car had finished")
add_step("empty rows of cars that did not start", rows_before, empty_non_starter.sum(), "removed",
         "lap row with no times for a driver classified as did not start")
laps = laps[~(empty_after_flag | empty_non_starter)].reset_index(drop=True)

after_last_lap = laps["lap_number"] > laps["classified_laps"]
laps["incomplete_lap"] = after_last_lap & laps["dnf"]
laps["beyond_classified"] = after_last_lap & ~laps["dnf"]
laps["disqualified"] = laps["dsq"]
laps["no_lap_time"] = laps["lap_duration"].isna()
add_step("retirement laps", len(laps), laps["incomplete_lap"].sum(), "flagged incomplete_lap",
         "the lap on which a car retired; it was started but not completed")
add_step("timed laps beyond the classified laps", len(laps), laps["beyond_classified"].sum(),
         "flagged beyond_classified", "lap after the chequered flag; never a racing lap")
add_step("laps of disqualified drivers", len(laps), laps["disqualified"].sum(), "flagged disqualified",
         "driver disqualified; the results give no number of laps")
add_step("laps without a lap time", len(laps), laps["no_lap_time"].sum(), "flagged no_lap_time",
         "lap_duration missing at the source (mostly lap 1 and retirement laps)")

print("\nTimed laps beyond the classified laps:")
print(laps.loc[laps["beyond_classified"], ["year", "circuit", "driver", "lap_number", "lap_duration",
                                           "classified_laps", "race_laps"]].to_string(index=False))

Rows after the driver's last classified lap: 300
  empty rows after the chequered flag (removed): 43
  empty rows of cars that did not start (removed): 15
 year     circuit driver  lap_number  classified_laps
 2023       Monza    TSU           1              0.0
 2023  Interlagos    LEC           1              0.0
 2024 Silverstone    GAS           1              0.0
 2024  Interlagos    STR           1              0.0
 2025   Melbourne    HAD           1              0.0
 2025       Monza    HUL           1              0.0
 2025 Silverstone    COL           1              0.0
 2025   Spielberg    SAI           1              0.0
 2026   Melbourne    HUL           1              0.0
 2026   Melbourne    PIA           1              0.0
 2026    Shanghai    BOR           1              0.0
 2026    Shanghai    PIA           1              0.0
 2026    Montreal    LIN           1              0.0
 2026  Interlagos    BOT           1              0.0
 2026   Las Vegas    ALB           

### Laps missing at the source

For some cars the lap file has fewer rows than the results say they completed. OpenF1 simply does not have these laps (checked by asking the API again). `race_missing_laps` is the number of such laps in the race; it is 0 for a complete race. Later stages decide whether to use the affected races.

In [6]:
per_car = laps.groupby(KEY).agg(classified_laps=("classified_laps", "first"), last_lap=("lap_number", "max"))
# Disqualified drivers have no classified laps; their own last lap is used instead.
per_car["expected"] = per_car["classified_laps"].fillna(per_car["last_lap"])
completed = laps[~laps["incomplete_lap"] & ~laps["beyond_classified"]].groupby(KEY).size()
per_car["rows"] = completed.reindex(per_car.index).fillna(0)
per_car["missing"] = (per_car["expected"] - per_car["rows"]).astype(int)
assert (per_car["missing"] >= 0).all(), "a car has more completed laps than the results say"

race_missing = per_car.groupby(level="session_key")["missing"].sum()
laps["race_missing_laps"] = laps["session_key"].map(race_missing).astype(int)
incomplete_races = races[races["session_key"].isin(race_missing[race_missing > 0].index)].copy()
incomplete_races["missing_laps"] = incomplete_races["session_key"].map(race_missing)
incomplete_races["cars_affected"] = incomplete_races["session_key"].map(
    per_car[per_car["missing"] > 0].groupby(level="session_key").size())
print(incomplete_races[["year", "circuit", "session_key", "missing_laps", "cars_affected"]].to_string(index=False))
add_step("laps in races with laps missing at the source", len(laps), (laps["race_missing_laps"] > 0).sum(),
         "flagged race_missing_laps > 0",
         f"OpenF1 has no rows for {int(race_missing.sum())} completed laps in these races")

 year circuit  session_key  missing_laps  cars_affected
 2023   Monza         9157             7              7
 2025   Miami        10033           446             19

## 4. Lap end time

The lap end is the next lap's `date_start`. When there is no next lap, or the next lap's start is not later than this lap's start (lap 1 of races restarted after an early red flag carries the restart time), it is `date_start + lap_duration`.

Missing lap-1 start times stay empty (flag `date_start_missing`); the end of those laps still comes from lap 2. A lap is flagged `time_mismatch` when the next lap's start differs from `date_start + lap_duration` by more than 0.5 s.

In [7]:
by_car = laps.groupby(KEY)
next_lap_number = by_car["lap_number"].shift(-1)
next_start = by_car["date_start"].shift(-1)
# Only the lap that really follows counts (2025 Miami jumps from lap 1 to lap 25).
next_start = next_start.where(next_lap_number == laps["lap_number"] + 1)

own_end = laps["date_start"] + pd.to_timedelta(laps["lap_duration"], unit="s")
goes_backwards = next_start <= laps["date_start"]
laps["lap_end"] = next_start.where(~goes_backwards).fillna(own_end)

difference = (next_start - own_end).dt.total_seconds()
laps["time_mismatch"] = (difference.abs() > TIME_MISMATCH_SECONDS) | goes_backwards
laps["date_start_missing"] = laps["date_start"].isna()

print("laps without date_start:", int(laps["date_start_missing"].sum()),
      "| all on lap 1:", bool((laps.loc[laps["date_start_missing"], "lap_number"] == 1).all()))
print("laps whose next lap starts earlier:", int(goes_backwards.sum()))
print(f"laps flagged time_mismatch (more than {TIME_MISMATCH_SECONDS} s):", int(laps["time_mismatch"].sum()))
print("laps without a lap end time:", int(laps["lap_end"].isna().sum()))
add_step("laps without a start time", len(laps), laps["date_start_missing"].sum(), "flagged date_start_missing",
         "date_start missing at the source (lap 1 only); left empty")
add_step("lap end time disagrees with the lap time", len(laps), laps["time_mismatch"].sum(), "flagged time_mismatch",
         f"next lap's start differs from date_start + lap_duration by more than {TIME_MISMATCH_SECONDS} s, "
         "or is not later than date_start")

laps without date_start: 52 | all on lap 1: True
laps whose next lap starts earlier: 21
laps flagged time_mismatch (more than 0.5 s): 261
laps without a lap end time: 261


## 5. Red-flag periods

A red-flag period runs from a `SESSION ABORTED` message to the next `SESSION STARTED` message in race control. Laps that overlap such a period are flagged `red_flag_lap`: their lap time includes the stoppage.

In [8]:
status = race_control[race_control["category"] == "SessionStatus"].sort_values(["session_key", "date"])
periods = []
for session_key, messages in status.groupby("session_key"):
    aborted_at = None
    for message in messages.itertuples():
        if message.message == "SESSION ABORTED":
            aborted_at = message.date
        elif message.message == "SESSION STARTED" and aborted_at is not None:
            periods.append({"session_key": session_key, "red_start": aborted_at, "red_end": message.date})
            aborted_at = None
red_flags = pd.DataFrame(periods).merge(races[["session_key", "year", "circuit"]], on="session_key")
print(red_flags[["year", "circuit", "red_start", "red_end"]].to_string(index=False))


def overlaps_red_flag(table, start_column, end_column):
    """Return True for rows whose time window overlaps a red-flag period of the same race."""
    result = pd.Series(False, index=table.index)
    for period in red_flags.itertuples():
        result |= ((table["session_key"] == period.session_key)
                   & (table[start_column] <= period.red_end)
                   & (table[end_column] >= period.red_start))
    return result


laps["red_flag_lap"] = overlaps_red_flag(laps, "date_start", "lap_end")
print("\nlaps overlapping a red-flag period:", int(laps["red_flag_lap"].sum()))
print("laps longer than 300 s:", int((laps["lap_duration"] > 300).sum()),
      "| of which flagged red_flag_lap:", int(((laps["lap_duration"] > 300) & laps["red_flag_lap"]).sum()))
add_step("laps during a red flag", len(laps), laps["red_flag_lap"].sum(), "flagged red_flag_lap",
         "the lap overlaps a red-flag period, so its time includes the stoppage")

 year     circuit                        red_start                          red_end
 2023   Melbourne 2023-04-02 05:17:02.518000+00:00 2023-04-02 05:33:00.429000+00:00
 2023   Melbourne 2023-04-02 06:41:07.569000+00:00 2023-04-02 06:56:00.419000+00:00
 2023   Melbourne 2023-04-02 07:00:26.572000+00:00 2023-04-02 07:33:00.456000+00:00
 2023   Zandvoort 2023-08-27 14:31:15.895000+00:00 2023-08-27 15:14:00.131000+00:00
 2023 Mexico City 2023-10-29 20:51:52.582000+00:00 2023-10-29 21:13:00.310000+00:00
 2023  Interlagos 2023-11-05 17:06:35.195000+00:00 2023-11-05 17:31:00.366000+00:00
 2024      Suzuka 2024-04-07 05:04:24.473000+00:00 2024-04-07 05:32:00.407000+00:00
 2024 Monte Carlo 2024-05-26 13:04:07.561000+00:00 2024-05-26 13:44:00.121000+00:00
 2024  Interlagos 2024-11-03 16:37:21.752000+00:00 2024-11-03 17:02:00.390000+00:00
 2026 Monte Carlo 2026-06-07 14:35:06.971000+00:00 2026-06-07 15:12:00.071000+00:00
 2026   Zandvoort 2026-08-23 13:05:27.956000+00:00 2026-08-23 13:33:00.08800

## 6. Stops: in-laps and out-laps

**Pit records (85 races):** a pit record is a stop. Its `lap_number` is the in-lap and the next lap is the out-lap (the pit record's `date` is the pit exit time).
- Records timed before the scheduled race start are removed.
- When the lap running at the record's timestamp is not the lap after its `lap_number`, and that running lap is flagged as an out-lap in the lap file, the lap number is wrong: the in-lap is then taken from the timestamp (flag `lap_from_date`).

**Out-lap flags (every race):** an out-lap flag (`is_pit_out_lap`, lap 2 or later) with no pit record on the lap before is also a stop, if it is confirmed by a stint change (checked in section 7). The in-lap is the lap before, and the stop has no durations. This gives the stops of the six 2023 races without pit data, and a few stops that the pit data of other races misses.

The `is_pit_out_lap` flag on lap 1 is ignored: it is set for the whole grid in two races.

In [9]:
print("is_pit_out_lap on lap 1 (ignored):")
print(laps[(laps["lap_number"] == 1) & laps["is_pit_out_lap"]].groupby(["year", "circuit"]).size().to_string())

pit = pit_raw.rename(columns={"lap_number": "in_lap", "date": "pit_date"})
# Lap 1 of a restarted race carries the restart time, so the scheduled start
# from the race index is used to find records from before the race.
scheduled_start = pd.to_datetime(races.set_index("session_key")["date"], format="ISO8601", utc=True)
before_race = pit["pit_date"] < pit["session_key"].map(scheduled_start)
print("\npit records:", len(pit), "| timed before the scheduled race start (removed):", int(before_race.sum()),
      "| of which with a duration:", int(pit.loc[before_race, "pit_duration"].notna().sum()))
pit = pit[~before_race].reset_index(drop=True)


def add_lap_window(stops):
    """Add the start of the in-lap and the end of the out-lap (the other lap is used when one is missing)."""
    stops = stops.drop(columns=["window_start", "window_end"], errors="ignore")
    lap_times = laps[LAP_KEY + ["date_start", "lap_end"]]
    in_lap = lap_times.rename(columns={"lap_number": "in_lap", "date_start": "in_start", "lap_end": "in_end"})
    out_lap = lap_times.rename(columns={"lap_number": "out_lap", "date_start": "out_start", "lap_end": "out_end"})
    stops = stops.assign(out_lap=stops["in_lap"] + 1)
    stops = stops.merge(in_lap, on=KEY + ["in_lap"], how="left").merge(out_lap, on=KEY + ["out_lap"], how="left")
    stops["window_start"] = stops["in_start"].fillna(stops["out_start"])
    stops["window_end"] = stops["out_end"].fillna(stops["in_end"])
    return stops.drop(columns=["in_start", "in_end", "out_start", "out_end"])


# The pit record's date is the pit exit, so it lies in the out-lap. Find the
# lap that was running at that time and compare it with the record's lap number.
running_lap = []
running_is_out_lap = []
for record in pit.itertuples():
    car_laps = laps[(laps["session_key"] == record.session_key) & (laps["driver_number"] == record.driver_number)
                    & (laps["date_start"] <= record.pit_date)]
    if len(car_laps) == 0:
        running_lap.append(np.nan)
        running_is_out_lap.append(False)
    else:
        running = car_laps.loc[car_laps["date_start"].idxmax()]
        running_lap.append(running["lap_number"])
        running_is_out_lap.append(bool(running["is_pit_out_lap"]))
pit["lap_at_pit_date"] = running_lap
contradicts = pit["lap_at_pit_date"].notna() & (pit["lap_at_pit_date"] != pit["in_lap"] + 1)
# The timestamp is trusted only when the lap file confirms it with an out-lap
# flag. Without that, the lap number is kept (laps missing at the source and
# restarted first laps make the timestamp comparison unreliable).
pit["lap_from_date"] = contradicts & pd.Series(running_is_out_lap, index=pit.index)
pit["in_lap_delivered"] = pit["in_lap"]
pit.loc[pit["lap_from_date"], "in_lap"] = pit.loc[pit["lap_from_date"], "lap_at_pit_date"].astype(int) - 1
pit = add_lap_window(pit)

with_names = pit.merge(races[["session_key", "year", "circuit"]], on="session_key")
print("pit records whose lap number contradicts the timestamp, by race:")
print(with_names[contradicts.values].groupby(["year", "circuit", "lap_from_date"]).size().to_string())
print("\nin-lap taken from the timestamp (lap_from_date):", int(pit["lap_from_date"].sum()))
print(with_names.loc[pit["lap_from_date"].values, ["year", "circuit", "driver_number", "in_lap_delivered", "in_lap",
                                                   "pit_date", "pit_duration"]].to_string(index=False))
assert not pit.duplicated(KEY + ["in_lap"]).any(), "two pit records on the same lap of one car"
pit["source"] = "pit"

is_pit_out_lap on lap 1 (ignored):
year  circuit    
2024  Montreal       20
2026  Mexico City    22

pit records: 3202 | timed before the scheduled race start (removed): 9 | of which with a duration: 0


pit records whose lap number contradicts the timestamp, by race:
year  circuit     lap_from_date
2023  Interlagos  False             2
2024  Suzuka      False            18
2025  Catalunya   False             1
2026  Melbourne   True             10

in-lap taken from the timestamp (lap_from_date): 10
 year   circuit  driver_number  in_lap_delivered  in_lap                         pit_date  pit_duration
 2026 Melbourne              1                46      11 2026-03-08 04:19:43.406000+00:00        18.266
 2026 Melbourne             10                46      11 2026-03-08 04:19:54.135000+00:00        18.672
 2026 Melbourne             31                46      11 2026-03-08 04:19:56.923000+00:00        18.570
 2026 Melbourne             55                46      11 2026-03-08 04:20:04.992000+00:00        19.859
 2026 Melbourne             30                46      11 2026-03-08 04:20:20.833000+00:00        19.112
 2026 Melbourne              5                18      33 2026-03-08 04:53:

In [10]:
# Stops from the out-lap flag, where no pit record covers them.
races_with_pit = set(pit_raw["session_key"])
races_without_pit = races[~races["session_key"].isin(races_with_pit)]
print("races without pit data:")
print(races_without_pit[["year", "circuit", "session_key"]].to_string(index=False))

flagged_out_laps = laps[laps["is_pit_out_lap"] & (laps["lap_number"] > 1)]
flag_stops = flagged_out_laps[KEY].copy()
flag_stops["in_lap"] = flagged_out_laps["lap_number"] - 1
has_pit_record = flag_stops.merge(pit[KEY + ["in_lap"]], on=KEY + ["in_lap"], how="left", indicator=True)["_merge"] == "both"
flag_stops = flag_stops[~has_pit_record.values]
print("\nout-lap flags without a pit record on the lap before:", len(flag_stops),
      "| in races with pit data:", int(flag_stops["session_key"].isin(races_with_pit).sum()))
flag_stops["source"] = "out_lap_flag"
flag_stops["lap_from_date"] = False
flag_stops = add_lap_window(flag_stops)

stops = pd.concat([pit, flag_stops], ignore_index=True)
stops["out_lap"] = stops["in_lap"] + 1

# A stop is a red-flag stop when its in-lap to out-lap window, or the time
# the car spent in the pit lane, overlaps a red-flag period.
stops["pit_entry"] = stops["pit_date"] - pd.to_timedelta(stops["pit_duration"], unit="s")
stops["red_flag"] = (overlaps_red_flag(stops, "window_start", "window_end")
                     | overlaps_red_flag(stops, "pit_entry", "pit_date"))
print("\nstops by source and red flag:")
print(stops.groupby(["source", "red_flag"]).size().to_string())

races without pit data:
 year     circuit  session_key
 2023      Sakhir         7953
 2023      Jeddah         7779
 2023   Melbourne         7787
 2023        Baku         9070
 2023       Miami         9078
 2023 Monte Carlo         9094

out-lap flags without a pit record on the lap before: 225 | in races with pit data: 6

stops by source and red flag:
source        red_flag
out_lap_flag  False        169
              True          56
pit           False       2959
              True         234


## 7. Stints: compound and tyre age

Each change of stint is matched to a stop of the same car:
1. normally the stop's in-lap is the lap **before** the new stint;
2. at Montreal, Melbourne and Monte Carlo the stop happens before the timing line, and the stints data starts the new stint **on the in-lap**. There the new stint is moved one lap later, to the out-lap, as at every other circuit (flag `stint_shifted`).

After this, stints that cover no lap (a set fitted and removed without a completed lap) are dropped from the lap assignment and counted.

A stint change is the `same_set` when the compound is unchanged and the tyre age simply carries on. This happens when the field is led through the pit lane behind the safety car, during red flags, and at stops without a tyre change.

In [11]:
stints = stints_raw.sort_values(KEY + ["stint_number"]).reset_index(drop=True)
by_car = stints.groupby(KEY)
previous_compound = by_car["compound"].shift(1)
previous_age = by_car["tyre_age_at_start"].shift(1)
previous_start = by_car["lap_start"].shift(1)
stints["is_change"] = stints["stint_number"] > 1
stints["same_set"] = ((stints["compound"] == previous_compound)
                      & (stints["tyre_age_at_start"] == previous_age + stints["lap_start"] - previous_start))
stints["lap_start_delivered"] = stints["lap_start"]
print("stints:", len(stints), "| stint changes:", int(stints["is_change"].sum()),
      "| covering no lap as delivered:", int((stints["lap_end"] < stints["lap_start"]).sum()))

# Match every stint change to a stop, going through each car's stints in order.
stop_laps = stops.groupby(KEY)["in_lap"].apply(set).to_dict()
matched_in_lap = []
for car, car_stints in stints.groupby(KEY, sort=False):
    free_stops = set(stop_laps.get(car, set()))
    for stint in car_stints.itertuples():
        match = np.nan
        if stint.is_change:
            if stint.lap_start - 1 in free_stops:
                match = stint.lap_start - 1
            elif stint.lap_start in free_stops:
                match = stint.lap_start
            if not np.isnan(match):
                free_stops.remove(match)
        matched_in_lap.append(match)
stints["stop_in_lap"] = matched_in_lap
stints["stint_shifted"] = stints["stop_in_lap"] == stints["lap_start"]

# Move the shifted stints to the out-lap; each stint then ends where the next one starts.
stints["lap_start"] = stints["lap_start"] + stints["stint_shifted"].astype(int)
next_stint_start = stints.groupby(KEY)["lap_start"].shift(-1)
stints["lap_end"] = (next_stint_start - 1).fillna(stints["lap_end"]).astype(int)

shifted = stints[stints["stint_shifted"]].merge(races[["session_key", "year", "circuit"]], on="session_key")
print("\nstints moved one lap later (stint_shifted):", len(shifted))
print(shifted.groupby("circuit").size().sort_values(ascending=False).to_string())

no_lap = stints["lap_end"] < stints["lap_start"]
print("\nstints covering no lap after the move (not assigned to any lap):", int(no_lap.sum()))
stints = stints[~no_lap].reset_index(drop=True)
stints["is_first"] = stints.groupby(KEY).cumcount() == 0

stints: 5301 | stint changes: 3445 | covering no lap as delivered: 43



stints moved one lap later (stint_shifted): 584
circuit
Melbourne       205
Montreal        189
Monte Carlo     185
Kuala Lumpur      5

stints covering no lap after the move (not assigned to any lap): 11


### Stint changes without a stop

A stint change with no stop on the lap before or on the same lap is listed as an exception:
- `red_flag`: the change falls in a red-flag period (tyres changed in the pit lane during the stoppage, no pit record);
- `no_stop_found`: the stints data puts the boundary on a lap where the car did not stop. The data is kept as delivered and the laps between this boundary and the car's nearest stop without a stint change are flagged `compound_uncertain` (the whole new stint when the car has no such stop). The stop gets the same flag in the pit-stop table.

In [12]:
exceptions = stints[~stints["is_first"] & stints["stop_in_lap"].isna()].copy()
exceptions["in_lap"] = exceptions["lap_start"] - 1
exceptions = add_lap_window(exceptions)
exceptions["reason"] = np.where(overlaps_red_flag(exceptions, "window_start", "window_end"),
                                "red_flag", "no_stop_found")

# Stops that were not matched to any stint change.
matched_stops = stints.loc[stints["stop_in_lap"].notna(), KEY + ["stop_in_lap", "same_set", "stint_shifted"]]
matched_stops = matched_stops.rename(columns={"stop_in_lap": "in_lap"})
stops = stops.merge(matched_stops, on=KEY + ["in_lap"], how="left", validate="one_to_one")
stops["has_stint_change"] = stops["same_set"].notna()
stops["stint_shifted"] = stops["stint_shifted"].eq(True)
stops["compound_uncertain"] = ~stops["has_stint_change"] & ~stops["red_flag"]

# A stop taken from an out-lap flag must be confirmed by a stint change.
unconfirmed = (stops["source"] == "out_lap_flag") & ~stops["has_stint_change"]
print("out-lap flags without a pit record and without a stint change (not counted as stops):",
      int(unconfirmed.sum()))
stops = stops[~unconfirmed].reset_index(drop=True)

# Laps between a misplaced boundary and the nearest unmatched stop of the same car.
first_flagged, last_flagged, paired_stop = [], [], []
for change in exceptions.itertuples():
    first_lap, last_lap, stop_lap = np.nan, np.nan, np.nan
    if change.reason == "no_stop_found":
        first_lap, last_lap = change.lap_start, change.lap_end
        candidates = stops[(stops["session_key"] == change.session_key)
                           & (stops["driver_number"] == change.driver_number)
                           & stops["compound_uncertain"]]
        if len(candidates) > 0:
            nearest = (candidates["out_lap"] - change.lap_start).abs().idxmin()
            stop_lap = candidates.loc[nearest, "in_lap"]
            first_lap = min(change.lap_start, stop_lap + 1)
            last_lap = max(change.lap_start, stop_lap + 1) - 1
    first_flagged.append(first_lap)
    last_flagged.append(last_lap)
    paired_stop.append(stop_lap)
exceptions["nearest_unmatched_stop_in_lap"] = paired_stop
exceptions["first_uncertain_lap"] = first_flagged
exceptions["last_uncertain_lap"] = last_flagged

exceptions = exceptions.merge(races[["session_key", "year", "circuit"]], on="session_key")
exceptions = exceptions.merge(driver_names, on=KEY, how="left")
exception_columns = ["year", "circuit", "session_key", "driver_number", "driver", "stint_number", "lap_start",
                     "lap_end", "compound", "tyre_age_at_start", "same_set", "reason",
                     "nearest_unmatched_stop_in_lap", "first_uncertain_lap", "last_uncertain_lap"]
exceptions = exceptions[exception_columns].sort_values(["year", "session_key", "driver_number", "stint_number"])
exceptions = exceptions.reset_index(drop=True)
print("stint changes without a matching stop:", len(exceptions))
print(exceptions.groupby("reason").size().to_string())
print("\nby race (changes, of which paired with a stop that has no stint change):")
print(exceptions.groupby(["year", "circuit", "reason"]).agg(
    changes=("lap_start", "size"), paired=("nearest_unmatched_stop_in_lap", "count")).to_string())
print("\nstops without a stint change:", int((~stops["has_stint_change"]).sum()),
      "| of which during a red flag:", int((~stops["has_stint_change"] & stops["red_flag"]).sum()),
      "| flagged compound_uncertain:", int(stops["compound_uncertain"].sum()))

out-lap flags without a pit record and without a stint change (not counted as stops): 0
stint changes without a matching stop: 87
reason
no_stop_found    87

by race (changes, of which paired with a stop that has no stint change):
                                      changes  paired
year circuit           reason                        
2023 Montreal          no_stop_found        1       1
2024 Lusail            no_stop_found        2       0
     Spa-Francorchamps no_stop_found        4       0
2025 Las Vegas         no_stop_found        1       0
     Melbourne         no_stop_found        6       6
     Miami             no_stop_found       15      15
     Spa-Francorchamps no_stop_found       20       6
2026 Austin            no_stop_found        1       1
     Catalunya         no_stop_found        1       1
     Hungaroring       no_stop_found        1       1
     Interlagos        no_stop_found        1       1
     Kuala Lumpur      no_stop_found       27      26
     Las Vega

### One row per stint lap, joined to the laps

`tyre_age` is the number of laps already done on the set when the lap starts, including laps from earlier sessions (`tyre_age_at_start`). `used_set` says that the set was not new when it was fitted.

Laps without tyre information keep an empty compound and tyre age, and `tyre_flag` says why: `no_stint` (no stint covers the lap) or `unknown_compound` (the stint's compound is `UNKNOWN`). `tyre_flag` is `compound_uncertain` for the laps described above; they keep the delivered compound.

In [13]:
stints["laps_in_stint"] = stints["lap_end"] - stints["lap_start"] + 1
stint_laps = stints.loc[stints.index.repeat(stints["laps_in_stint"])].copy()
laps_into_stint = stint_laps.groupby(level=0).cumcount()
stint_laps["lap_number"] = stint_laps["lap_start"] + laps_into_stint
stint_laps["tyre_age"] = stint_laps["tyre_age_at_start"] + laps_into_stint
stint_laps["used_set"] = stint_laps["tyre_age_at_start"] > 0

# One car has two stints claiming the same lap; the later stint is kept
# because it is the later message of the tyre data.
overlapping = stint_laps.duplicated(LAP_KEY, keep="last")
print("stint laps claimed by two stints (the later stint is kept):", int(overlapping.sum()))
print(stint_laps.loc[stint_laps.duplicated(LAP_KEY, keep=False),
                     LAP_KEY + ["stint_number", "compound", "tyre_age"]].to_string(index=False))
stint_laps = stint_laps[~overlapping]
stint_laps = stint_laps[LAP_KEY + ["stint_number", "compound", "tyre_age", "used_set", "stint_shifted"]]

laps = laps.merge(stint_laps, on=LAP_KEY, how="left", validate="one_to_one")

laps["tyre_flag"] = ""
laps.loc[laps["stint_number"].isna(), "tyre_flag"] = "no_stint"
unknown_compound = laps["compound"] == "UNKNOWN"
laps.loc[unknown_compound, "tyre_flag"] = "unknown_compound"
laps.loc[unknown_compound, "compound"] = None
for change in exceptions[exceptions["reason"] == "no_stop_found"].itertuples():
    in_range = ((laps["session_key"] == change.session_key) & (laps["driver_number"] == change.driver_number)
                & laps["lap_number"].between(change.first_uncertain_lap, change.last_uncertain_lap)
                & (laps["tyre_flag"] == ""))
    laps.loc[in_range, "tyre_flag"] = "compound_uncertain"

laps["stint_number"] = laps["stint_number"].astype("Int64")
laps["tyre_age"] = laps["tyre_age"].astype("Int64")
laps["used_set"] = laps["used_set"].astype("boolean")
laps["stint_shifted"] = laps["stint_shifted"].eq(True)

print("\ncompound:")
print(laps["compound"].value_counts(dropna=False).to_string())
print("\ntyre_flag:")
print(laps["tyre_flag"].replace("", "(none)").value_counts().to_string())
add_step("stints covering no lap", len(laps), 0, "none",
         f"{int(no_lap.sum())} stints (a set fitted and removed without a completed lap) are assigned to no lap")
add_step("stint moved to the out-lap", len(laps), laps["stint_shifted"].sum(), "flagged stint_shifted",
         f"{len(shifted)} stints started on the in-lap in the stints data (stop before the timing line)")
add_step("laps without a stint", len(laps), (laps["tyre_flag"] == "no_stint").sum(), "flagged tyre_flag = no_stint",
         "no stint covers the lap (almost all are retirement laps); compound and tyre age empty")
add_step("laps with an unknown compound", len(laps), (laps["tyre_flag"] == "unknown_compound").sum(),
         "flagged tyre_flag = unknown_compound", "the stints data says UNKNOWN; compound set to empty")
add_step("laps with an uncertain compound", len(laps), (laps["tyre_flag"] == "compound_uncertain").sum(),
         "flagged tyre_flag = compound_uncertain",
         f"{int((exceptions['reason'] == 'no_stop_found').sum())} stint changes on a lap without a stop; "
         "compound and tyre age kept as delivered")

stint laps claimed by two stints (the later stint is kept): 6
 session_key  driver_number  lap_number  stint_number     compound  tyre_age
        9693              5           3             2 INTERMEDIATE         2
        9693              5           3             4 INTERMEDIATE         3
        9693              5           4             4 INTERMEDIATE         4
        9693              5           4             6       MEDIUM         0
        9693             30           2             1 INTERMEDIATE         1
        9693             30           3             2 INTERMEDIATE         0
        9693             30           2             4 INTERMEDIATE         2
        9693             30           3             5 INTERMEDIATE         3
        9693             87           3             2 INTERMEDIATE         2
        9693             87           3             4 INTERMEDIATE         0
        9693             87           4             4 INTERMEDIATE         1
        9693  


compound:
compound
HARD            45245
MEDIUM          35737
SOFT            15898
INTERMEDIATE     4753
NaN               209
WET                84
None                4

tyre_flag:
tyre_flag
(none)                100587
compound_uncertain      1130
no_stint                 209
unknown_compound           4


## 8. Mark in-laps and out-laps

Every stop marks its in-lap and its out-lap, including stops during a red flag (those laps also carry `red_flag_lap`).

In [14]:
in_laps = stops[KEY + ["in_lap"]].rename(columns={"in_lap": "lap_number"}).assign(is_in_lap=True)
out_laps = stops[KEY + ["out_lap"]].rename(columns={"out_lap": "lap_number"}).assign(is_out_lap=True)
laps = laps.merge(in_laps, on=LAP_KEY, how="left", validate="one_to_one")
laps = laps.merge(out_laps, on=LAP_KEY, how="left", validate="one_to_one")
laps["is_in_lap"] = laps["is_in_lap"].eq(True)
laps["is_out_lap"] = laps["is_out_lap"].eq(True)
add_step("in-laps", len(laps), laps["is_in_lap"].sum(), "flagged is_in_lap", "the lap that ends with a stop")
add_step("out-laps", len(laps), laps["is_out_lap"].sum(), "flagged is_out_lap", "the lap that follows a stop")

# Compare with the out-lap flag of the lap file (lap 2 or later, races with pit data).
flagged = laps[laps["is_pit_out_lap"] & (laps["lap_number"] > 1) & laps["session_key"].isin(races_with_pit)]
print("is_pit_out_lap flags after lap 1 in races with pit data:", len(flagged),
      "| with a stop on the lap before:", int(flagged["is_out_lap"].sum()))
print(flagged[~flagged["is_out_lap"]].groupby(["year", "circuit"]).size().to_string())
melbourne_2026 = flagged[(flagged["year"] == 2026) & (flagged["circuit"] == "Melbourne")]
print(f"2026 Melbourne: {len(melbourne_2026)} out-lap flags, {int(melbourne_2026['is_out_lap'].sum())} with a stop")
ours_only = laps[laps["is_out_lap"] & ~laps["is_pit_out_lap"]]
print("out-laps from stops that the lap file does not flag:", len(ours_only),
      "| of which the lap row exists but belongs to a red-flag stop:", int(ours_only["red_flag_lap"].sum()))

is_pit_out_lap flags after lap 1 in races with pit data: 3195 | with a stop on the lap before: 3195
Series([], )
2026 Melbourne: 32 out-lap flags, 32 with a stop
out-laps from stops that the lap file does not flag: 0 | of which the lap row exists but belongs to a red-flag stop: 0


## 9. Pit-stop table

One row per stop. `tyre_age_at_stop` is the number of laps on the old set when it comes off (its tyre age at the start of the in-lap plus one). The duration fields are empty for stops taken from the out-lap flag (the six races without pit data and a few others), and `stop_duration` only exists from 2024 on.

In [15]:
tyres_by_lap = stint_laps[LAP_KEY + ["compound", "tyre_age"]]
before = tyres_by_lap.rename(columns={"lap_number": "in_lap", "compound": "compound_before", "tyre_age": "age_in_lap"})
after = tyres_by_lap.rename(columns={"lap_number": "out_lap", "compound": "compound_after",
                                     "tyre_age": "tyre_age_after"})
pit_stops = stops.merge(before, on=KEY + ["in_lap"], how="left").merge(after, on=KEY + ["out_lap"], how="left")
pit_stops["tyre_age_at_stop"] = (pit_stops["age_in_lap"] + 1).astype("Int64")
pit_stops["tyre_age_after"] = pit_stops["tyre_age_after"].astype("Int64")
pit_stops["same_set"] = pit_stops["same_set"].astype("boolean")
pit_stops = pit_stops.merge(races[["session_key", "year", "circuit"]], on="session_key", how="left")
pit_stops = pit_stops.merge(driver_names, on=KEY, how="left", validate="many_to_one")
pit_stops = pit_stops[["year", "session_key", "circuit", "driver_number", "driver", "team", "in_lap", "out_lap",
                       "source", "pit_date", "pit_duration", "lane_duration", "stop_duration",
                       "compound_before", "compound_after", "tyre_age_at_stop", "tyre_age_after",
                       "has_stint_change", "same_set", "red_flag", "stint_shifted", "lap_from_date",
                       "compound_uncertain"]]
pit_stops = pit_stops.sort_values(["year", "session_key", "driver_number", "in_lap"]).reset_index(drop=True)

print("stops:", len(pit_stops))
print(pit_stops.groupby(["year", "source"]).size().unstack(fill_value=0).to_string())
print("\nflags (number of stops):")
print(pit_stops[["has_stint_change", "red_flag", "stint_shifted", "lap_from_date", "compound_uncertain"]].sum().to_string())
print("same_set:", int(pit_stops["same_set"].sum()))
print("\nmissing values:")
print(pit_stops.isna().sum()[lambda counts: counts > 0].to_string())

stops: 3418
source  out_lap_flag  pit
year                     
2023             219  693
2024               0  825
2025               0  821
2026               6  854

flags (number of stops):
has_stint_change      3347
red_flag               290
stint_shifted          574
lap_from_date           10
compound_uncertain      71
same_set: 266

missing values:
pit_date          225
pit_duration      225
lane_duration     225
stop_duration    1953
same_set           71


Same-set stint changes outside red flags, by race. Where the whole field has them, race control shows the safety car leading the cars through the pit lane: the cars pass the pit lane without stopping, so the set and its tyre age carry on.

In [16]:
same_set_stops = pit_stops[pit_stops["same_set"].fillna(False) & ~pit_stops["red_flag"]]
summary = same_set_stops.groupby(["year", "circuit"]).agg(
    stops=("in_lap", "size"), cars=("driver_number", "nunique"),
    first_in_lap=("in_lap", "min"), last_in_lap=("in_lap", "max"),
    median_pit_duration=("pit_duration", "median"), with_stop_duration=("stop_duration", "count"))
print(summary.to_string())
through_pit_lane = race_control[race_control["message"].str.contains("THROUGH THE PIT LANE", na=False)]
through_pit_lane = through_pit_lane.merge(races[["session_key", "year", "circuit"]], on="session_key")
print("\nrace control messages about the pit lane:")
print(through_pit_lane[["year", "circuit", "date", "message"]].to_string(index=False))

                         stops  cars  first_in_lap  last_in_lap  median_pit_duration  with_stop_duration
year circuit                                                                                            
2023 Sakhir                  1     1            15           15                  NaN                   0
     Silverstone             1     1            36           36               34.300                   0
     Spielberg              19    19             2            2               16.700                   0
     Suzuka                  1     1             2            2               32.700                   0
2024 Imola                   1     1            23           23               38.400                   0
     Las Vegas               1     1            11           11               15.700                   0
     Lusail                 34    17            35           47               26.150                   1
     Miami                   1     1            31     

## 10. Safety-car and virtual-safety-car periods

Race control announces a safety car with `SAFETY CAR DEPLOYED` and a virtual safety car (VSC) with `VIRTUAL SAFETY CAR DEPLOYED` (`VSC DEPLOYED` in 2026). The messages `SAFETY CAR IN THIS LAP` and `VSC ENDING` only announce the end. The period really ends with the next `TRACK CLEAR`, or earlier when the race is red-flagged, the chequered flag falls, or a VSC is upgraded to a safety car.

A second `SAFETY CAR DEPLOYED` while a safety car is already out (2026 Silverstone) continues the same period.

In [17]:
DEPLOYED = {"SAFETY CAR DEPLOYED": "safety_car", "VIRTUAL SAFETY CAR DEPLOYED": "vsc", "VSC DEPLOYED": "vsc"}
PERIOD_ENDS = {"TRACK CLEAR": "track clear", "SESSION ABORTED": "red flag", "CHEQUERED FLAG": "chequered flag"}

# The race control file of 2026 Mexico City also holds messages from the day
# before, so only messages from the scheduled race start on are used.
during_race = race_control["date"] >= race_control["session_key"].map(scheduled_start)
messages = race_control[during_race & race_control["message"].isin(list(DEPLOYED) + list(PERIOD_ENDS))]
messages = messages.sort_values(["session_key", "date"], kind="stable")

periods = []
for session_key, race_messages in messages.groupby("session_key"):
    current = None
    for message in race_messages.itertuples():
        kind = DEPLOYED.get(message.message)
        ended_by = PERIOD_ENDS.get(message.message)
        if kind == "safety_car" and current is not None and current["kind"] == "vsc":
            ended_by = "safety car"  # a VSC upgraded to a safety car
        if current is not None and ended_by is not None:
            current["end"] = message.date
            current["ended_by"] = ended_by
            periods.append(current)
            current = None
        if kind is not None and current is None:
            current = {"session_key": session_key, "kind": kind, "start": message.date}
    if current is not None:
        # No end message at all: the period runs to the last race control message.
        last_message = race_control.loc[race_control["session_key"] == session_key, "date"].max()
        current["end"] = last_message
        current["ended_by"] = "end of messages"
        periods.append(current)
periods = pd.DataFrame(periods).merge(races[["session_key", "year", "circuit"]], on="session_key")
periods["seconds"] = (periods["end"] - periods["start"]).dt.total_seconds()

print("deployment messages:", int(messages["message"].isin(list(DEPLOYED)).sum()), "| periods:", len(periods))
print("\nperiods by kind and by what ended them:")
print(periods.groupby(["kind", "ended_by"]).size().unstack(fill_value=0).to_string())
print("\nperiods per season:")
print(periods.groupby(["year", "kind"]).size().unstack(fill_value=0).to_string())
print("\nlength of the periods in seconds:")
print(periods.groupby("kind")["seconds"].describe().round(0).to_string())

deployment messages: 119 | periods: 118

periods by kind and by what ended them:
ended_by    chequered flag  red flag  safety car  track clear
kind                                                         
safety_car               2         8           0           51
vsc                      2         1           4           50

periods per season:
kind  safety_car  vsc
year                 
2023          16   10
2024          13    9
2025          17   13
2026          15   25

length of the periods in seconds:
            count   mean    std   min    25%    50%    75%     max
kind                                                              
safety_car   61.0  491.0  220.0  54.0  372.0  466.0  611.0  1036.0
vsc          57.0  134.0   75.0  20.0   79.0  117.0  185.0   325.0


## 11. Track status and restart laps

`track_status` of a lap is `red`, `safety_car`, `vsc` or `green`. A lap is marked when **any part of the car's own lap** (from its start to its end) overlaps a period, so cars that are not on the leader's lap are handled correctly. When a lap overlaps more than one kind of period, red wins over safety car, and safety car over VSC. The red-flag laps are the ones found in section 5.

`restart_lap` is the first green lap of a car directly after a safety-car lap or a red-flag lap: the field is bunched up (or starts from the grid). It is not set after a VSC, where the gaps are kept and the pace is normal straight away.

A few laps have no start time (lap 1 in some races) or no end time (retirement laps). They are judged on the part of the lap that is known; the 19 laps with no time at all (lap 1 of 2025 Miami) stay `green`.

Race control has no message for the end of the laps that are driven behind the safety car when a race is resumed after a red flag (or started behind the safety car). Those laps therefore stay `green` here; the first of them is the `restart_lap`, and the outlier rule in section 16 keeps the slow ones out of the pace model.

In [18]:
def overlaps_any(table, periods_table, start_column, end_column):
    """Return True for rows whose time window overlaps one of the periods of the same race."""
    result = pd.Series(False, index=table.index)
    for period in periods_table.itertuples():
        result |= ((table["session_key"] == period.session_key)
                   & (table[start_column] <= period.end)
                   & (table[end_column] >= period.start))
    return result


# The window of a lap. A missing start (lap 1 in a few races) is rebuilt from
# the end and the lap time, and when the lap time is missing too the lap counts
# as its end only. A lap without an end (retirement) counts as its start only.
rebuilt_start = laps["lap_end"] - pd.to_timedelta(laps["lap_duration"], unit="s")
laps["window_start"] = laps["date_start"].fillna(rebuilt_start).fillna(laps["lap_end"])
laps["window_end"] = laps["lap_end"].fillna(laps["date_start"])
print("laps without a start time:", int(laps["date_start"].isna().sum()),
      "| start rebuilt from end and lap time:", int((laps["date_start"].isna() & rebuilt_start.notna()).sum()),
      "| only the end is known:", int((laps["date_start"].isna() & rebuilt_start.isna() & laps["lap_end"].notna()).sum()),
      "| no time at all:", int(laps["window_start"].isna().sum()))
print("laps without an end time (only the start is known):", int((laps["lap_end"].isna() & laps["date_start"].notna()).sum()))
print()

laps["under_safety_car"] = overlaps_any(laps, periods[periods["kind"] == "safety_car"], "window_start", "window_end")
laps["under_vsc"] = overlaps_any(laps, periods[periods["kind"] == "vsc"], "window_start", "window_end")
laps["track_status"] = np.select([laps["red_flag_lap"], laps["under_safety_car"], laps["under_vsc"]],
                                 ["red", "safety_car", "vsc"], default="green")

laps = laps.sort_values(LAP_KEY).reset_index(drop=True)
by_car = laps.groupby(KEY)
previous_status = by_car["track_status"].shift(1)
follows_directly = by_car["lap_number"].shift(1) == laps["lap_number"] - 1
laps["restart_lap"] = ((laps["track_status"] == "green") & previous_status.isin(["safety_car", "red"])
                       & follows_directly)

# The first two green laps of a car after a red flag: the lap behind the safety
# car or extra formation lap, and the standing or rolling start. Race control
# does not mark them, and section 15 does not count stops on them as strategy stops.
status_two_back = by_car["track_status"].shift(2)
two_back_follows = by_car["lap_number"].shift(2) == laps["lap_number"] - 2
first_after_red = (laps["track_status"] == "green") & (previous_status == "red") & follows_directly
second_after_red = ((laps["track_status"] == "green") & (previous_status == "green") & follows_directly
                    & (status_two_back == "red") & two_back_follows)
laps["after_red_flag"] = first_after_red | second_after_red
print("first two green laps after a red flag:", int(laps["after_red_flag"].sum()),
      "| first:", int(first_after_red.sum()), "| second:", int(second_after_red.sum()))
print()

print("laps by track status:")
print(laps.groupby(["year", "track_status"]).size().unstack(fill_value=0).assign(
    total=lambda table: table.sum(axis=1)).to_string())
print("\nlaps touching both a safety car and a VSC (marked safety_car):", int((laps["under_safety_car"] & laps["under_vsc"]).sum()))
print("red-flag laps that also touch a safety car or VSC (marked red):",
      int((laps["red_flag_lap"] & (laps["under_safety_car"] | laps["under_vsc"])).sum()))
print("\nrestart laps:", int(laps["restart_lap"].sum()),
      "| after a safety car:", int((laps["restart_lap"] & (previous_status == "safety_car")).sum()),
      "| after a red flag:", int((laps["restart_lap"] & (previous_status == "red")).sum()))

add_step("laps under a safety car", len(laps), (laps["track_status"] == "safety_car").sum(),
         "flagged track_status = safety_car",
         f"any part of the car's lap overlaps one of the {int((periods['kind'] == 'safety_car').sum())} safety-car periods")
add_step("laps under a virtual safety car", len(laps), (laps["track_status"] == "vsc").sum(),
         "flagged track_status = vsc",
         f"any part of the car's lap overlaps one of the {int((periods['kind'] == 'vsc').sum())} VSC periods")
add_step("restart laps", len(laps), laps["restart_lap"].sum(), "flagged restart_lap",
         "first green lap of a car directly after a safety-car or red-flag lap")

laps without a start time: 52 | start rebuilt from end and lap time: 0 | only the end is known: 33 | no time at all: 19
laps without an end time (only the start is known): 242



first two green laps after a red flag: 423 | first: 223 | second: 200

laps by track status:
track_status  green  red  safety_car  vsc  total
year                                            
2023          22979  160         989  311  24439
2024          25316   86         966  256  26624
2025          24278    0        1471  511  26260
2026          22201  149        1286  971  24607

laps touching both a safety car and a VSC (marked safety_car): 72
red-flag laps that also touch a safety car or VSC (marked red): 159

restart laps: 1061 | after a safety car: 838 | after a red flag: 223


How each period maps to laps: the number of cars and laps it touches and the first and last lap number among them. `one_block` says that for every car the marked laps are consecutive.

In [19]:
rows = []
for period in periods.itertuples():
    touched = laps[(laps["session_key"] == period.session_key) & (laps["window_start"] <= period.end)
                   & (laps["window_end"] >= period.start)]
    per_car = touched.groupby("driver_number")["lap_number"].agg(["min", "max", "size"])
    rows.append({"cars": len(per_car), "laps": len(touched), "first_lap": touched["lap_number"].min(),
                 "last_lap": touched["lap_number"].max(),
                 "one_block": bool((per_car["max"] - per_car["min"] + 1 == per_car["size"]).all())})
periods = pd.concat([periods.drop(columns=["cars", "laps", "first_lap", "last_lap", "one_block"], errors="ignore"),
                     pd.DataFrame(rows)], axis=1)
periods[["first_lap", "last_lap"]] = periods[["first_lap", "last_lap"]].astype("Int64")
shown = periods.assign(start=periods["start"].dt.strftime("%H:%M:%S"), end=periods["end"].dt.strftime("%H:%M:%S"))
print(shown[["year", "circuit", "kind", "start", "end", "ended_by", "cars", "laps", "first_lap", "last_lap",
             "one_block"]].to_string(index=False))

 year            circuit       kind    start      end       ended_by  cars  laps  first_lap  last_lap  one_block
 2023             Jeddah safety_car 17:31:10 17:38:27    track clear    19    71         17        20       True
 2023          Melbourne safety_car 05:04:27 05:09:12    track clear    19    57          1         3       True
 2023          Melbourne safety_car 05:14:24 05:17:02       red flag    18    36          7         8       True
 2023          Melbourne        vsc 05:48:36 05:50:52    track clear    17    34         18        19       True
 2023          Melbourne safety_car 06:38:32 06:41:07       red flag    16    34         53        55       True
 2023             Sakhir        vsc 16:10:19 16:12:08    track clear    18    31         40        42       True
 2023               Baku safety_car 11:22:24 11:29:44    track clear    19    69         10        13       True
 2023           Montreal        vsc 18:12:28 18:13:14    track clear    19    30          7     

## 12. Weather per lap

The weather file has one record per minute. Each lap takes the record nearest in time to the start of the lap (to its end when the start is unknown): track temperature and air temperature (degrees Celsius) and `rainfall` (0 or 1).

In [20]:
weather_columns = ["track_temperature", "air_temperature", "rainfall"]
laps = laps.drop(columns=weather_columns + ["weather_date"], errors="ignore")
weather_records = weather.rename(columns={"date": "weather_date"}).sort_values("weather_date")

# merge_asof needs sorted, non-missing times; laps without any time keep empty weather.
has_time = laps["window_start"].notna()
with_time = laps.loc[has_time, LAP_KEY + ["window_start"]].sort_values("window_start")
nearest = pd.merge_asof(with_time, weather_records, left_on="window_start", right_on="weather_date",
                        by="session_key", direction="nearest")
laps = laps.merge(nearest[LAP_KEY + weather_columns + ["weather_date"]], on=LAP_KEY, how="left",
                  validate="one_to_one")

distance = (laps["window_start"] - laps["weather_date"]).dt.total_seconds().abs()
print("weather records:", len(weather), "| races with weather:", weather["session_key"].nunique(), "of", len(races))
print("seconds between the lap start and the nearest weather record:")
print(distance.describe().round(1).to_string())
no_weather = laps["track_temperature"].isna()
print("\nlaps without weather (no start and no end time):", int(no_weather.sum()))
print(laps[no_weather].groupby(["year", "circuit", "lap_number"]).size().to_string())
print("\nlaps starting with rainfall = 1:", int((laps["rainfall"] == 1).sum()))
add_step("laps without weather", len(laps), no_weather.sum(), "weather left empty",
         "the lap has no start and no end time, so no nearest weather record")

weather records: 14600 | races with weather: 91 of 91
seconds between the lap start and the nearest weather record:
count    101911.0
mean         15.0
std           8.7
min           0.0
25%           7.5
50%          14.9
75%          22.5
max          58.9

laps without weather (no start and no end time): 19
year  circuit  lap_number
2025  Miami    1             19

laps starting with rainfall = 1: 2863


## 13. Wet races

A race is wet if any car used intermediate or wet tyres on any lap. Wet races are excluded from the pit-strategy analysis. The flag is copied to every lap and every stop of the race.

The `rainfall` field is shown next to it for comparison only: it does not decide anything, because a track can be wet without rain falling and rain can fall without anyone needing wet tyres.

In [21]:
on_wet_tyres = laps["compound"].isin(WET_COMPOUNDS)
wet_sessions = set(laps.loc[on_wet_tyres, "session_key"])
laps["wet_race"] = laps["session_key"].isin(wet_sessions)
pit_stops["wet_race"] = pit_stops["session_key"].isin(wet_sessions)

wet_summary = laps[laps["wet_race"]].assign(on_wet_tyres=on_wet_tyres).groupby(["year", "circuit"]).agg(
    laps=("lap_number", "size"), wet_tyre_laps=("on_wet_tyres", "sum"), laps_with_rainfall=("rainfall", "sum"))
wet_summary["wet_tyre_share"] = (wet_summary["wet_tyre_laps"] / wet_summary["laps"]).round(3)
wet_summary["laps_with_rainfall"] = wet_summary["laps_with_rainfall"].astype(int)
print("wet races:", len(wet_sessions), "| dry races:", len(races) - len(wet_sessions))
print(wet_summary.to_string())

rain_laps = laps[~laps["wet_race"]].groupby(["year", "circuit"])["rainfall"].sum()
print("\ndry races with rainfall = 1 at the start of at least one lap (number of such laps):")
print(rain_laps[rain_laps > 0].astype(int).to_string())
add_step("laps in wet races", len(laps), laps["wet_race"].sum(), "flagged wet_race",
         f"{len(wet_sessions)} races in which a car used intermediate or wet tyres")

wet races: 11 | dry races: 80
                        laps  wet_tyre_laps  laps_with_rainfall  wet_tyre_share
year circuit                                                                   
2023 Monte Carlo        1515            442                 293           0.292
     Zandvoort          1344            327                 289           0.243
2024 Interlagos         1136           1134                 833           0.998
     Montreal           1272            844                 324           0.664
     Silverstone         961            234                 411           0.243
2025 Melbourne           926            681                 292           0.735
     Silverstone         826            607                  98           0.735
     Spa-Francorchamps   879             20                   0           0.023
2026 Interlagos         1436            514                  11           0.358
     Kuala Lumpur       1145             20                   0           0.017
     Montr

## 14. Pit loss

**Per stop.** The time lost by stopping is the in-lap time plus the out-lap time, minus the driver's typical lap before the stop and the typical lap after it. The typical lap is the median of up to 3 *normal green laps* in the 3 laps before the in-lap (or after the out-lap): green track status, not an in-lap or out-lap, not lap 1, with a lap time. Summing in-lap and out-lap handles Monte Carlo, Melbourne and Montreal, where the stop sits inside the in-lap. A stop without a normal green lap on one of the two sides gets no pit loss.

**Per race.** The median over the *usable stops* of the race. A usable stop has a tyre change, is not `compound_uncertain`, is not during a red flag, has a green in-lap and a green out-lap, is not on lap 1, and is not a slow stop.

A slow stop is a problem at the stop or a served penalty, which says nothing about the normal cost of stopping. It is found within its own race, because the pit loss of a circuit can change between seasons (Singapore from 2025, Shanghai in 2026):
- in the 85 races with pit data, a stop whose pit-lane time is more than 5 s above the median pit-lane time of the race;
- in the six 2023 races without pit data, a stop whose lap-based pit loss is more than 8 s above the median pit loss of the race.

Both medians are taken over the stops of the race that meet the other conditions. The rule applies to all races, wet ones included. A stop without a pit-lane time in a race with pit data cannot be judged and stays.

A race with fewer than 3 usable stops takes the median over the usable stops at the same circuit in the dry races of all seasons (`pit_loss_source` = `circuit`).

The pit-lane time of the pit data (`lane_duration`, 85 races) is kept as a comparison. It is the time between the pit entry and exit lines. It is not the time lost: a car that stays out also needs time for that stretch of track, while braking before the entry line, accelerating after the exit line and the out-lap on cold tyres are not in it.

In [22]:
# In-lap and out-lap of every stop: track status, lap time and whether the car retired on it.
pit_stops = pit_stops.drop(columns=["in_lap_status", "in_lap_time", "in_lap_incomplete", "in_lap_after_red_flag",
                                    "out_lap_status", "out_lap_time", "out_lap_incomplete",
                                    "out_lap_after_red_flag"], errors="ignore")
lap_facts = laps[LAP_KEY + ["track_status", "lap_duration", "incomplete_lap", "after_red_flag"]]
for side in ["in_lap", "out_lap"]:
    facts = lap_facts.rename(columns={"lap_number": side, "track_status": f"{side}_status",
                                      "lap_duration": f"{side}_time", "incomplete_lap": f"{side}_incomplete",
                                      "after_red_flag": f"{side}_after_red_flag"})
    pit_stops = pit_stops.merge(facts, on=KEY + [side], how="left", validate="many_to_one")

normal_green = ((laps["track_status"] == "green") & ~laps["is_in_lap"] & ~laps["is_out_lap"]
                & (laps["lap_number"] > 1) & laps["lap_duration"].notna())
normal_laps = laps.loc[normal_green, LAP_KEY + ["lap_duration"]]


def typical_lap(stop_table, lap_column, offsets):
    """Return the median time of the driver's normal green laps at the given distances from a stop lap."""
    times = {}
    for offset in offsets:
        wanted = stop_table[KEY].assign(lap_number=stop_table[lap_column] + offset)
        times[offset] = wanted.merge(normal_laps, on=LAP_KEY, how="left")["lap_duration"].to_numpy()
    return pd.DataFrame(times, index=stop_table.index).median(axis=1)


window = range(1, TYPICAL_LAP_WINDOW + 1)
pit_stops["typical_lap_before"] = typical_lap(pit_stops, "in_lap", [-offset for offset in window])
pit_stops["typical_lap_after"] = typical_lap(pit_stops, "out_lap", list(window))
pit_stops["pit_loss"] = (pit_stops["in_lap_time"] + pit_stops["out_lap_time"]
                         - pit_stops["typical_lap_before"] - pit_stops["typical_lap_after"])

pit_stops["tyre_change"] = pit_stops["has_stint_change"] & ~pit_stops["same_set"].fillna(False).astype(bool)
candidate = (pit_stops["tyre_change"] & ~pit_stops["compound_uncertain"] & ~pit_stops["red_flag"]
             & (pit_stops["in_lap_status"] == "green") & (pit_stops["out_lap_status"] == "green")
             & (pit_stops["in_lap"] > 1))
with_loss = candidate & pit_stops["pit_loss"].notna()

# Slow stops, judged against the other stops of the same race: on the pit-lane
# time where the race has pit data, on the lap-based pit loss in the six races without.
race_medians = pit_stops[with_loss].groupby("session_key")[["lane_duration", "pit_loss"]].median()
above_lane_median = pit_stops["lane_duration"] - pit_stops["session_key"].map(race_medians["lane_duration"])
above_loss_median = pit_stops["pit_loss"] - pit_stops["session_key"].map(race_medians["pit_loss"])
has_pit_data = pit_stops["session_key"].isin(races_with_pit)
slow_by_lane_time = with_loss & has_pit_data & (above_lane_median > SLOW_STOP_LANE_TIME)
slow_by_pit_loss = with_loss & ~has_pit_data & (above_loss_median > SLOW_STOP_PIT_LOSS)
pit_stops["pit_loss_slow"] = slow_by_lane_time | slow_by_pit_loss
pit_stops["pit_loss_usable"] = with_loss & ~pit_stops["pit_loss_slow"]
usable = pit_stops[pit_stops["pit_loss_usable"]]
print("stops:", len(pit_stops), "| tyre change, green in-lap and out-lap, no red flag, not lap 1:", int(candidate.sum()),
      "| of which with a pit loss:", int(with_loss.sum()))
print(f"left out as slow stops: pit-lane time more than {SLOW_STOP_LANE_TIME} s above the race median:",
      int(slow_by_lane_time.sum()),
      f"| pit loss more than {SLOW_STOP_PIT_LOSS} s above the race median (six races without pit data):",
      int(slow_by_pit_loss.sum()))
print("slow stops left out in total:", int(pit_stops["pit_loss_slow"].sum()),
      "| in dry races:", int((pit_stops["pit_loss_slow"] & ~pit_stops["wet_race"]).sum()),
      "| in wet races:", int((pit_stops["pit_loss_slow"] & pit_stops["wet_race"]).sum()),
      "| usable stops:", len(usable))
print("slow stops left out, per season:", pit_stops[pit_stops["pit_loss_slow"]].groupby("year").size().to_dict())
print("stops with a pit loss but no pit-lane time in races with pit data (cannot be judged, kept):",
      int((with_loss & has_pit_data & pit_stops["lane_duration"].isna()).sum()))
print("candidates without a pit loss: no in-lap or out-lap time:",
      int((candidate & (pit_stops["in_lap_time"].isna() | pit_stops["out_lap_time"].isna())).sum()),
      "| no normal green lap before:", int((candidate & pit_stops["typical_lap_before"].isna()).sum()),
      "| no normal green lap after:", int((candidate & pit_stops["typical_lap_after"].isna()).sum()))

print("\nusable stops in dry races, seconds:")
dry_usable = usable[~usable["wet_race"]]
print(dry_usable[["pit_loss", "lane_duration", "stop_duration"]].describe().round(2).to_string())
both = dry_usable.dropna(subset=["pit_loss", "lane_duration"])
print("pit-lane time minus pit loss, per stop (seconds):")
print((both["lane_duration"] - both["pit_loss"]).describe().round(2).to_string())

stops: 3418 | tyre change, green in-lap and out-lap, no red flag, not lap 1: 2158 | of which with a pit loss: 2085
left out as slow stops: pit-lane time more than 5 s above the race median: 106 | pit loss more than 8 s above the race median (six races without pit data): 18
slow stops left out in total: 124 | in dry races: 95 | in wet races: 29 | usable stops: 1961
slow stops left out, per season: {2023: 37, 2024: 30, 2025: 29, 2026: 28}
stops with a pit loss but no pit-lane time in races with pit data (cannot be judged, kept): 2
candidates without a pit loss: no in-lap or out-lap time: 1 | no normal green lap before: 22 | no normal green lap after: 51

usable stops in dry races, seconds:
       pit_loss  lane_duration  stop_duration
count   1709.00        1638.00         780.00
mean      23.61          23.46           2.94
std       26.31           2.70           0.94
min      -24.20          17.00           1.80
25%       20.89          21.82           2.40
50%       22.44          23

In [23]:
per_race = usable.groupby("session_key").agg(pit_loss_stops=("pit_loss", "size"), own_pit_loss=("pit_loss", "median"),
                                             pit_lane_time=("lane_duration", "median"))
circuit_pit_loss = usable[~usable["wet_race"]].groupby("circuit")["pit_loss"].median()

race_pit_loss = races[["session_key", "year", "circuit"]].merge(per_race, on="session_key", how="left")
race_pit_loss["pit_loss_stops"] = race_pit_loss["pit_loss_stops"].fillna(0).astype(int)
enough_stops = race_pit_loss["pit_loss_stops"] >= MIN_STOPS_FOR_PIT_LOSS
race_pit_loss["pit_loss"] = race_pit_loss["own_pit_loss"].where(enough_stops,
                                                                race_pit_loss["circuit"].map(circuit_pit_loss))
race_pit_loss["pit_loss_source"] = np.where(enough_stops, "race", "circuit")

print("races with their own estimate:", int(enough_stops.sum()), "| with the circuit median:", int((~enough_stops).sum()))
print("\nraces that take the circuit median (own_pit_loss is what their own few stops would give):")
print(race_pit_loss.loc[~enough_stops, ["year", "circuit", "pit_loss_stops", "own_pit_loss", "pit_loss"]].round(2).to_string(index=False))
print("\nraces without a pit loss estimate:", int(race_pit_loss["pit_loss"].isna().sum()))

races with their own estimate: 85 | with the circuit median: 6

races that take the circuit median (own_pit_loss is what their own few stops would give):
 year    circuit  pit_loss_stops  own_pit_loss  pit_loss
 2023  Melbourne               2         21.45     20.87
 2024 Interlagos               1         25.29     22.24
 2025      Miami               1         18.07     20.01
 2026      Monza               2         31.76     25.30
 2026       Baku               2         25.50     21.54
 2026  Las Vegas               0           NaN     22.49

races without a pit loss estimate: 0


## 15. Strategy stops

Only strategy stops are used in the tests of H1 to H3. A stop is a strategy stop when **none** of the following holds; `not_strategy_reason` gives the first one that does, in this order:

1. `red_flag`: the stop is during a red flag;
2. `no_tyre_change`: no stint change, or the same set carries on (`same_set`, mostly safety-car passages through the pit lane);
3. `compound_uncertain`: the stop cannot be paired with a stint change;
4. `wet_race`: the race is wet;
5. `in_lap_not_green`: the in-lap is under a safety car or VSC (or has no lap row). Only the in-lap counts: when the track is green on the in-lap, the decision to stop was taken under green, even if a safety car comes out afterwards;
6. `lap_1`: the stop is at the end of lap 1 (damage, not strategy);
7. `restart_after_red_flag`: the in-lap is one of the first two green laps after a red flag, the lap behind the safety car or extra formation lap and the standing or rolling start (the same logic as lap 1);
8. `retirement`: the car retired on the in-lap or the out-lap, or has no out-lap;
9. `slow_stop`: the stop loses more than 20 s more than the pit loss of its race (likely repairs). A stop with a 5 or 10 s penalty stays. A stop without a pit loss (no normal green lap on one side) cannot be judged and stays.

In [24]:
race_loss = race_pit_loss.set_index("session_key")["pit_loss"]
pit_stops["above_race_pit_loss"] = pit_stops["pit_loss"] - pit_stops["session_key"].map(race_loss)
retirement = (pit_stops["out_lap_status"].isna() | pit_stops["out_lap_incomplete"].eq(True)
              | pit_stops["in_lap_incomplete"].eq(True))
stop_conditions = [
    ("red_flag", pit_stops["red_flag"]),
    ("no_tyre_change", ~pit_stops["tyre_change"]),
    ("compound_uncertain", pit_stops["compound_uncertain"]),
    ("wet_race", pit_stops["wet_race"]),
    ("in_lap_not_green", pit_stops["in_lap_status"] != "green"),
    ("lap_1", pit_stops["in_lap"] == 1),
    ("restart_after_red_flag", pit_stops["in_lap_after_red_flag"].eq(True)),
    ("retirement", retirement),
    ("slow_stop", pit_stops["above_race_pit_loss"] > SLOW_STOP_FOR_STRATEGY),
]
pit_stops["not_strategy_reason"] = np.select([condition for _, condition in stop_conditions],
                                             [name for name, _ in stop_conditions], default="")
pit_stops["strategy_stop"] = pit_stops["not_strategy_reason"] == ""

stop_reasons = pd.DataFrame({
    "stops_with_this_condition": [int(condition.sum()) for _, condition in stop_conditions],
    "first_reason_for": [int((pit_stops["not_strategy_reason"] == name).sum()) for name, _ in stop_conditions],
}, index=[name for name, _ in stop_conditions])
print("stops:", len(pit_stops), "| strategy stops:", int(pit_stops["strategy_stop"].sum()))
print(stop_reasons.to_string())
print("\nstrategy stops without a pit loss (cannot be judged as slow, kept):",
      int((pit_stops["strategy_stop"] & pit_stops["pit_loss"].isna()).sum()))
slow = pit_stops[pit_stops["not_strategy_reason"] == "slow_stop"].sort_values("above_race_pit_loss", ascending=False)
print("\nten largest slow_stop cases (seconds):")
print(slow.head(10)[["year", "circuit", "driver", "in_lap", "lane_duration", "stop_duration", "in_lap_time",
                     "out_lap_time", "pit_loss", "above_race_pit_loss"]].round(1).to_string(index=False))
print("\nstrategy stops between 5 and 20 s above the pit loss of their race (kept):",
      int((pit_stops["strategy_stop"] & pit_stops["above_race_pit_loss"].between(5, SLOW_STOP_FOR_STRATEGY)).sum()))
print("\nstrategy stops by the track status of the out-lap:")
print(pit_stops.loc[pit_stops["strategy_stop"], "out_lap_status"].value_counts().to_string())
print("\nstrategy stops per season:")
print(pit_stops.groupby("year").agg(stops=("strategy_stop", "size"), strategy_stops=("strategy_stop", "sum"),
                                    races_with_one=("session_key", lambda keys: keys[
                                        pit_stops.loc[keys.index, "strategy_stop"]].nunique())).to_string())
add_step("pit stops that are not strategy stops (counts are stops, not laps)", len(pit_stops),
         (~pit_stops["strategy_stop"]).sum(), "flagged strategy_stop = False",
         "red flag, no tyre change, uncertain compound, wet race, in-lap not green, lap 1, first two laps after "
         "a red flag, retirement or slow stop; "
         "first reason in not_strategy_reason")

stops: 3418 | strategy stops: 1873
                        stops_with_this_condition  first_reason_for
red_flag                                      290               290
no_tyre_change                                337               265
compound_uncertain                             71                 0
wet_race                                      568               449
in_lap_not_green                             1113               492
lap_1                                         111                17
restart_after_red_flag                          1                 1
retirement                                      8                 1
slow_stop                                     423                30

strategy stops without a pit loss (cannot be judged as slow, kept): 76

ten largest slow_stop cases (seconds):
 year   circuit driver  in_lap  lane_duration  stop_duration  in_lap_time  out_lap_time  pit_loss  above_race_pit_loss
 2026 Melbourne    STR      34            NaN         

## 16. Clean laps for the pace model

A clean lap is a normal racing lap that shows the pace of car and tyre. A lap is clean when **none** of the following holds; `not_clean_reason` gives the first one that does, in this order: `lap_1`, `no_lap_time`, `red_flag`, `safety_car`, `vsc`, `restart`, `in_lap`, `out_lap`, `no_compound`, `compound_uncertain`, `outlier`.

**Outlier rule.** Among the laps that pass everything else, a lap is an outlier when it is slower than 1.07 times the driver's median lap in the same stint (the median over those remaining laps). There is no rule for fast laps. This removes laps with an incident, and the slow laps behind the safety car after a red flag that race control does not mark.

`clean_lap` does not depend on the weather: laps on wet tyres can be clean. Stage 3 uses `wet_race` to keep dry races only.

In [25]:
lap_conditions = [
    ("lap_1", laps["lap_number"] == 1),
    ("no_lap_time", laps["lap_duration"].isna()),
    ("red_flag", laps["track_status"] == "red"),
    ("safety_car", laps["track_status"] == "safety_car"),
    ("vsc", laps["track_status"] == "vsc"),
    ("restart", laps["restart_lap"]),
    ("in_lap", laps["is_in_lap"]),
    ("out_lap", laps["is_out_lap"]),
    ("no_compound", laps["compound"].isna()),
    ("compound_uncertain", laps["tyre_flag"] == "compound_uncertain"),
]
laps["not_clean_reason"] = np.select([condition for _, condition in lap_conditions],
                                     [name for name, _ in lap_conditions], default="")

# The driver's typical lap in the stint: the median over the laps that passed every condition above.
remaining = laps[laps["not_clean_reason"] == ""]
stint_median = remaining.groupby(KEY + ["stint_number"])["lap_duration"].transform("median")
ratio = remaining["lap_duration"] / stint_median
print("laps passing every condition before the outlier rule:", len(remaining))
print("lap time divided by the driver's median in the stint, quantiles:")
print(ratio.quantile([0.01, 0.05, 0.5, 0.95, 0.99, 0.999]).round(4).to_string())
for threshold in [1.05, OUTLIER_RATIO, 1.10]:
    print(f"slower than {threshold:.2f} x the stint median: {int((ratio > threshold).sum())} laps "
          f"({int((ratio[~remaining['wet_race']] > threshold).sum())} in dry races)")
laps.loc[ratio[ratio > OUTLIER_RATIO].index, "not_clean_reason"] = "outlier"
laps["clean_lap"] = laps["not_clean_reason"] == ""

reason_names = [name for name, _ in lap_conditions] + ["outlier"]
lap_reasons = pd.DataFrame({
    "laps_with_this_condition": [int(condition.sum()) for _, condition in lap_conditions]
                                + [int((laps["not_clean_reason"] == "outlier").sum())],
    "first_reason_for": [int((laps["not_clean_reason"] == name).sum()) for name in reason_names],
}, index=reason_names)
print(f"\nclean laps: {int(laps['clean_lap'].sum())} of {len(laps)} ({laps['clean_lap'].mean():.1%})")
print(lap_reasons.to_string())
assert not (laps["clean_lap"] & (laps["incomplete_lap"] | laps["beyond_classified"])).any()

add_step("laps that are not clean for the pace model", len(laps), (~laps["clean_lap"]).sum(),
         "flagged clean_lap = False",
         "lap 1, no lap time, red flag, safety car, VSC, restart, in-lap, out-lap, no or uncertain compound, "
         "or outlier; first reason in not_clean_reason")
add_step("of which outliers", len(laps), (laps["not_clean_reason"] == "outlier").sum(),
         "flagged not_clean_reason = outlier",
         f"slower than {OUTLIER_RATIO} times the driver's median lap in the same stint")

laps passing every condition before the outlier rule: 86512
lap time divided by the driver's median in the stint, quantiles:
0.010    0.9775
0.050    0.9898
0.500    1.0000
0.950    1.0181
0.990    1.0493
0.999    1.2897
slower than 1.05 x the stint median: 844 laps (275 in dry races)
slower than 1.07 x the stint median: 561 laps (191 in dry races)
slower than 1.10 x the stint median: 304 laps (133 in dry races)



clean laps: 85951 of 101930 (84.3%)
                    laps_with_this_condition  first_reason_for
lap_1                                   1839              1839
no_lap_time                              293               215
red_flag                                 395               358
safety_car                              4712              4482
vsc                                     2049              2013
restart                                 1061              1050
in_lap                                  3412              2272
out_lap                                 3414              2253
no_compound                              213                 1
compound_uncertain                      1130               935
outlier                                  561               561


## 17. Races table

One row per race. `safety_car_periods` and `vsc_periods` count the periods of section 10 (a VSC upgraded to a safety car counts once in each). `pit_loss` is the estimate of section 14 in seconds, `pit_loss_stops` the number of usable stops of the race (slow stops left out), and `pit_lane_time` the median pit-lane time of those stops where the pit data has it.

In [26]:
period_counts = periods.groupby(["session_key", "kind"]).size().unstack(fill_value=0)
race_table = races[["year", "session_key", "circuit", "date"]].rename(columns={"date": "scheduled_start"})
race_table["laps"] = race_table["session_key"].map(winner_laps).astype(int)
race_table["wet_race"] = race_table["session_key"].isin(wet_sessions)
race_table["safety_car_periods"] = race_table["session_key"].map(period_counts["safety_car"]).fillna(0).astype(int)
race_table["vsc_periods"] = race_table["session_key"].map(period_counts["vsc"]).fillna(0).astype(int)
race_table["red_flag"] = race_table["session_key"].isin(set(red_flags["session_key"]))
race_table["race_missing_laps"] = race_table["session_key"].map(race_missing).astype(int)
race_table = race_table.merge(
    race_pit_loss[["session_key", "pit_loss", "pit_loss_stops", "pit_loss_source", "pit_lane_time"]],
    on="session_key", how="left", validate="one_to_one")
race_table = race_table.sort_values(["year", "scheduled_start"]).drop(columns="scheduled_start").reset_index(drop=True)
print(race_table.round(2).to_string(index=False))

 year  session_key            circuit  laps  wet_race  safety_car_periods  vsc_periods  red_flag  race_missing_laps  pit_loss  pit_loss_stops pit_loss_source  pit_lane_time
 2023         7953             Sakhir    57     False                   0            1     False                  0     24.23              35            race            NaN
 2023         7779             Jeddah    50     False                   1            0     False                  0     20.23               7            race            NaN
 2023         7787          Melbourne    58     False                   3            1      True                  0     20.87               2         circuit            NaN
 2023         9070               Baku    51     False                   1            0     False                  0     22.26               6            race            NaN
 2023         9078              Miami    57     False                   0            0     False                  0     20.18          

## 18. Checks

In [27]:
# 1. Exactly one row per driver per lap.
assert not laps.duplicated(LAP_KEY).any()
print("1. one row per driver per lap:", len(laps), "rows,", laps[KEY].drop_duplicates().shape[0], "cars")

# 2. Every lap has a compound and tyre age, or a flag saying why not.
no_tyre = laps["compound"].isna() | laps["tyre_age"].isna()
assert (laps.loc[no_tyre, "tyre_flag"].isin(["no_stint", "unknown_compound"])).all()
assert not (laps.loc[~no_tyre, "tyre_flag"].isin(["no_stint", "unknown_compound"])).any()
print("2. laps without compound or tyre age:", int(no_tyre.sum()), "| all with a tyre_flag: True")
print(laps.loc[no_tyre].groupby(["tyre_flag", "incomplete_lap"]).size().to_string())

# 3. No driver has more laps than the winner; drivers with fewer laps are explained.
last_laps = laps.groupby(KEY).agg(year=("year", "first"), circuit=("circuit", "first"), driver=("driver", "first"),
                                  last_lap=("lap_number", "max"), race_laps=("race_laps", "first"),
                                  classified_laps=("classified_laps", "first"), dnf=("dnf", "first"),
                                  dsq=("dsq", "first"), race_missing_laps=("race_missing_laps", "first"))
assert (last_laps["last_lap"] <= last_laps["race_laps"]).all()
fewer = last_laps[last_laps["last_lap"] < last_laps["race_laps"]].copy()
fewer["why"] = np.select(
    [fewer["dnf"], fewer["dsq"], fewer["classified_laps"] < fewer["race_laps"], fewer["race_missing_laps"] > 0],
    ["retired", "disqualified", "lapped", "laps missing at the source"], default="unexplained")
print("\n3. cars with more laps than the winner: 0")
print("   cars with fewer lap rows than the winner:", len(fewer))
print(fewer["why"].value_counts().to_string())
print(fewer.loc[fewer["why"].isin(["laps missing at the source", "unexplained"]),
                ["year", "circuit", "driver", "last_lap", "classified_laps", "race_laps"]].to_string())
assert (fewer["why"] != "unexplained").all()

# 4. Every stint change has a matching stop, or is listed as an exception.
changes = stints[~stints["is_first"]]
print("\n4. stint changes (stints that cover a lap):", len(changes),
      "| with a stop:", int(changes["stop_in_lap"].notna().sum()),
      "| listed as exceptions:", len(exceptions))
assert changes["stop_in_lap"].notna().sum() + len(exceptions) == len(changes)

1. one row per driver per lap: 101930 rows, 1839 cars
2. laps without compound or tyre age: 213 | all with a tyre_flag: True
tyre_flag         incomplete_lap
no_stint          False               1
                  True              208
unknown_compound  False               2
                  True                2



3. cars with more laps than the winner: 0
   cars with fewer lap rows than the winner: 670
why
lapped                        425
retired                       237
laps missing at the source      7
disqualified                    1
                           year circuit driver  last_lap  classified_laps  race_laps
session_key driver_number                                                           
9157        2              2023   Monza    SAR        50             51.0         51
            10             2023   Monza    GAS        50             51.0         51
            18             2023   Monza    STR        50             51.0         51
            24             2023   Monza    ZHO        50             51.0         51
            40             2023   Monza    LAW        50             51.0         51
            77             2023   Monza    BOT        50             51.0         51
            81             2023   Monza    PIA        50             51.0         51

4.

Checks for part 2: every safety-car and VSC period is one block of consecutive laps for every car, shown for three races; every race has a pit loss estimate and the estimates are plausible.

In [28]:
# 5. Each period maps to a block of consecutive marked laps for every car.
print("5. periods:", len(periods), "| one block of consecutive laps for every car:", int(periods["one_block"].sum()),
      "| periods touching no lap:", int((periods["laps"] == 0).sum()))
print(periods.loc[periods["laps"] == 0, ["year", "circuit", "kind", "start", "end"]].to_string(index=False))
assert periods["one_block"].all()
# Every safety-car or VSC lap belongs to a period, and every period with laps has marked laps.
assert (laps["track_status"].isin(["safety_car", "vsc"])
        == ((laps["under_safety_car"] | laps["under_vsc"]) & ~laps["red_flag_lap"])).all()

EXAMPLE_RACES = [(2023, "Jeddah"), (2024, "Miami"), (2026, "Baku")]
for year, circuit in EXAMPLE_RACES:
    race_periods = periods[(periods["year"] == year) & (periods["circuit"] == circuit)]
    race_laps = laps[(laps["year"] == year) & (laps["circuit"] == circuit)]
    print(f"\n{year} {circuit}")
    print(race_periods.assign(start=race_periods["start"].dt.strftime("%H:%M:%S"),
                              end=race_periods["end"].dt.strftime("%H:%M:%S"))[
        ["kind", "start", "end", "ended_by", "first_lap", "last_lap"]].to_string(index=False))
    around = race_laps[race_laps["lap_number"].between(race_periods["first_lap"].min() - 2,
                                                       race_periods["last_lap"].max() + 2)]
    by_lap = around.groupby("lap_number").agg(
        leader_start=("date_start", "min"), cars=("driver_number", "size"),
        green=("track_status", lambda status: (status == "green").sum()),
        safety_car=("track_status", lambda status: (status == "safety_car").sum()),
        vsc=("track_status", lambda status: (status == "vsc").sum()),
        restart=("restart_lap", "sum"), median_lap_time=("lap_duration", "median"))
    by_lap["leader_start"] = by_lap["leader_start"].dt.strftime("%H:%M:%S")
    print(by_lap.round(1).to_string())

# 6. Every race has a pit loss estimate, and the estimates are plausible.
assert race_table["pit_loss"].notna().all()
print(f"\n6. races with a pit loss estimate: {int(race_table['pit_loss'].notna().sum())} of {len(race_table)}")
print(f"   range: {race_table['pit_loss'].min():.1f} to {race_table['pit_loss'].max():.1f} s"
      f" | dry races: {race_table.loc[~race_table['wet_race'], 'pit_loss'].min():.1f} to"
      f" {race_table.loc[~race_table['wet_race'], 'pit_loss'].max():.1f} s")
by_circuit = race_table[~race_table["wet_race"]].groupby("circuit").agg(
    dry_races=("pit_loss", "size"), pit_loss=("pit_loss", "median"), lowest=("pit_loss", "min"),
    highest=("pit_loss", "max"), pit_lane_time=("pit_lane_time", "median")).sort_values("pit_loss")
print("   five circuits with the lowest pit loss (median over their dry races, seconds):")
print(by_circuit.head(5).round(1).to_string())
print("   five circuits with the highest pit loss:")
print(by_circuit.tail(5).round(1).to_string())
print("   five lowest and five highest single races:")
ordered = race_table.sort_values("pit_loss")[["year", "circuit", "wet_race", "pit_loss", "pit_loss_stops",
                                              "pit_loss_source", "pit_lane_time"]]
print(pd.concat([ordered.head(5), ordered.tail(5)]).round(1).to_string(index=False))

# 7. Flags and reasons agree.
assert (laps["clean_lap"] == (laps["not_clean_reason"] == "")).all()
assert (pit_stops["strategy_stop"] == (pit_stops["not_strategy_reason"] == "")).all()
assert not pit_stops.loc[pit_stops["strategy_stop"], "wet_race"].any()
print("\n7. clean laps:", int(laps["clean_lap"].sum()), "| strategy stops:", int(pit_stops["strategy_stop"].sum()))

5. periods: 118 | one block of consecutive laps for every car: 118 | periods touching no lap: 1
 year circuit kind                     start                       end
 2025   Miami  vsc 2025-05-04 20:06:03+00:00 2025-05-04 20:09:08+00:00

2023 Jeddah
      kind    start      end    ended_by  first_lap  last_lap
safety_car 17:31:10 17:38:27 track clear         17        20
           leader_start  cars  green  safety_car  vsc  restart  median_lap_time
lap_number                                                                     
15             17:25:58    20     20           0    0        0             96.0
16             17:27:32    20     20           0    0        0             96.1
17             17:29:06    20      6          14    0        0            107.7
18             17:30:40    19      0          19    0        0            136.2
19             17:32:48    19      0          19    0        0            134.7
20             17:35:58    19      0          19    0        0   

           leader_start  cars  green  safety_car  vsc  restart  median_lap_time
lap_number                                                                     
20             20:32:53    20     20           0    0        0             93.7
21             20:34:25    20     20           0    0        0             93.7
22             20:35:58    20      6           0   14        0            100.3
23             20:37:31    20      0           0   20        0            107.3
24             20:39:22    20     20           0    0        0             93.6
25             20:40:54    20     20           0    0        0             93.3
26             20:42:27    20     20           0    0        0             93.5
27             20:43:59    20     20           0    0        0             93.5
28             20:45:36    20      1          19    0        0            111.1
29             20:47:08    19      0          19    0        0            137.6
30             20:49:21    19      0    

   range: 18.0 to 30.2 s | dry races: 18.0 to 30.2 s
   five circuits with the lowest pit loss (median over their dry races, seconds):
                   dry_races  pit_loss  lowest  highest  pit_lane_time
circuit                                                               
Spa-Francorchamps          3      18.8    18.0     21.9           23.5
Zandvoort                  3      19.6    18.5     23.2           18.0
Montreal                   2      19.9    19.7     20.1           23.7
Miami                      4      20.0    19.7     20.2           22.7
Silverstone                2      20.6    20.3     20.8           28.7
   five circuits with the highest pit loss:
           dry_races  pit_loss  lowest  highest  pit_lane_time
circuit                                                       
Monza              4      25.3    25.0     25.4           24.6
Madring            1      26.5    26.5     26.5           31.4
Singapore          4      27.0    23.4     30.2           27.0
Lusail   


7. clean laps: 85951 | strategy stops: 1873


## 19. Save

In [29]:
lap_columns = ["year", "session_key", "circuit", "driver_number", "driver", "team", "lap_number", "lap_duration",
               "duration_sector_1", "duration_sector_2", "duration_sector_3", "i1_speed", "i2_speed", "st_speed",
               "date_start", "lap_end", "is_pit_out_lap", "race_laps", "compound", "stint_number", "tyre_age",
               "used_set", "is_in_lap", "is_out_lap", "tyre_flag", "stint_shifted", "incomplete_lap",
               "beyond_classified", "disqualified", "no_lap_time", "date_start_missing", "time_mismatch",
               "red_flag_lap", "race_missing_laps", "track_status", "restart_lap", "track_temperature",
               "air_temperature", "rainfall", "wet_race", "clean_lap", "not_clean_reason"]
laps = laps[lap_columns].sort_values(["year", "session_key", "driver_number", "lap_number"]).reset_index(drop=True)

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
RESULTS.mkdir(parents=True, exist_ok=True)
laps.to_parquet(DATA_PROCESSED / "laps.parquet", index=False)
pit_stops = pit_stops.drop(columns=["in_lap_incomplete", "out_lap_incomplete", "in_lap_after_red_flag",
                                    "out_lap_after_red_flag"])
pit_stops.to_parquet(DATA_PROCESSED / "pit_stops.parquet", index=False)
race_table.to_parquet(DATA_PROCESSED / "races.parquet", index=False)
cleaning_table = pd.DataFrame(cleaning_steps)
cleaning_table.to_csv(RESULTS / "cleaning_table.csv", index=False)
exceptions.to_csv(RESULTS / "stint_change_exceptions.csv", index=False)
print("saved laps.parquet:", laps.shape, "| pit_stops.parquet:", pit_stops.shape,
      "| races.parquet:", race_table.shape)

saved laps.parquet: (101930, 42) | pit_stops.parquet: (3418, 37) | races.parquet: (91, 13)


## 20. Report

In [30]:
pd.set_option("display.max_colwidth", 120)
print("CLEANING TABLE")
print(cleaning_table.to_string(index=False))

print("\nPROBLEM COUNTS AFTER CLEANING")
flag_columns = ["incomplete_lap", "beyond_classified", "disqualified", "no_lap_time", "date_start_missing",
                "time_mismatch", "red_flag_lap", "stint_shifted", "is_in_lap", "is_out_lap"]
print("lap rows:", len(laps), "| races:", laps["session_key"].nunique(),
      "| duplicate (race, driver, lap):", int(laps.duplicated(LAP_KEY).sum()))
print(laps[flag_columns].sum().to_string())
print("tyre_flag:", laps["tyre_flag"].replace("", "(none)").value_counts().to_dict())
print("laps in races with missing laps:", laps[laps["race_missing_laps"] > 0].groupby(
    ["year", "circuit"])["race_missing_laps"].agg(["size", "first"]).rename(
    columns={"size": "lap_rows", "first": "missing_laps"}).to_dict("index"))
print("missing values in laps:")
print(laps.isna().sum()[lambda counts: counts > 0].to_string())
print("\nlaps per season:")
print(laps.groupby("year").agg(races=("session_key", "nunique"), laps=("lap_number", "size")).to_string())

print("\nPART 2: RACE SITUATION")
season = race_table.groupby("year").agg(
    races=("session_key", "size"), wet_races=("wet_race", "sum"), dry_races=("wet_race", lambda wet: (~wet).sum()),
    safety_car_periods=("safety_car_periods", "sum"), races_with_safety_car=("safety_car_periods", lambda n: (n > 0).sum()),
    vsc_periods=("vsc_periods", "sum"), races_with_vsc=("vsc_periods", lambda n: (n > 0).sum()),
    races_with_red_flag=("red_flag", "sum"))
season.loc["all"] = season.sum()
print(season.to_string())
print("\nwet races:", ", ".join(f"{race.year} {race.circuit}" for race in race_table[race_table["wet_race"]].itertuples()))

print("\nlaps by track status, restart laps and clean laps per season:")
lap_season = laps.groupby("year").agg(
    laps=("lap_number", "size"), safety_car=("track_status", lambda status: (status == "safety_car").sum()),
    vsc=("track_status", lambda status: (status == "vsc").sum()),
    red=("track_status", lambda status: (status == "red").sum()), restart=("restart_lap", "sum"),
    clean_laps=("clean_lap", "sum"))
lap_season.loc["all"] = lap_season.sum()
lap_season["clean_share"] = (lap_season["clean_laps"] / lap_season["laps"]).round(4)
print(lap_season.to_string())
print("\nclean-lap share per season, dry races only:")
print(laps[~laps["wet_race"]].groupby("year")["clean_lap"].agg(["size", "sum", "mean"]).rename(
    columns={"size": "laps", "sum": "clean_laps", "mean": "clean_share"}).round(4).to_string())
print("\nfirst reason a lap is not clean:")
print(laps["not_clean_reason"].replace("", "(clean)").value_counts().to_string())

print("\nstrategy stops per season:")
stop_season = pit_stops.groupby("year").agg(stops=("strategy_stop", "size"), strategy_stops=("strategy_stop", "sum"))
stop_season.loc["all"] = stop_season.sum()
print(stop_season.to_string())
print("\nfirst reason a stop is not a strategy stop:")
print(pit_stops["not_strategy_reason"].replace("", "(strategy stop)").value_counts().to_string())

print("\npit loss by circuit and season (seconds; * = wet race, c = circuit median used):")
marked = race_table.assign(value=race_table["pit_loss"].round(1).astype(str)
                           + np.where(race_table["wet_race"], "*", "")
                           + np.where(race_table["pit_loss_source"] == "circuit", "c", ""))
pit_loss_table = marked.pivot(index="circuit", columns="year", values="value").fillna("")
pit_loss_table["dry_median"] = race_table[~race_table["wet_race"]].groupby("circuit")["pit_loss"].median().round(1)
pit_loss_table["pit_lane_time"] = race_table.groupby("circuit")["pit_lane_time"].median().round(1)
print(pit_loss_table.sort_values("dry_median").to_string())

print("\nmissing values in pit_stops:")
print(pit_stops.isna().sum()[lambda counts: counts > 0].to_string())
print("missing values in races:")
print(race_table.isna().sum()[lambda counts: counts > 0].to_string())

print("\nEXAMPLE ROWS: laps")
print(laps.sample(3, random_state=42).T.to_string())
print("\nEXAMPLE ROWS: pit_stops")
print(pit_stops.sample(3, random_state=42).T.to_string())
print("\nEXAMPLE ROWS: races")
print(race_table.sample(3, random_state=42).T.to_string())

print(f"\nRunning time: {time.time() - start_time:.1f} seconds")

CLEANING TABLE
                                                              step  laps_before  laps_affected                                 action                                                                                                                                                                         reason
                            combine the raw lap files of all races       101988              0                                   none                                                                                                                                                                 starting point
                                              exact duplicate rows       101988              0                                removed                                                                                                                                                   the same row delivered twice
                               empty rows after the cheque

not_clean_reason
(clean)               85951
safety_car             4482
in_lap                 2272
out_lap                2253
vsc                    2013
lap_1                  1839
restart                1050
compound_uncertain      935
outlier                 561
red_flag                358
no_lap_time             215
no_compound               1

strategy stops per season:
      stops  strategy_stops
year                       
2023    912             515
2024    825             543
2025    821             480
2026    860             335
all    3418            1873

first reason a stop is not a strategy stop:
not_strategy_reason
(strategy stop)           1873
in_lap_not_green           492
wet_race                   449
red_flag                   290
no_tyre_change             265
slow_stop                   30
lap_1                       17
retirement                   1
restart_after_red_flag       1

pit loss by circuit and season (seconds; * = wet race, c = circuit median used

year                 2023    2024   2025   2026  dry_median  pit_lane_time
circuit                                                                   
Spa-Francorchamps    18.0    18.8  26.8*   21.9        18.8           23.6
Zandvoort           24.1*    23.2   18.5   19.6        19.6           19.0
Montreal             20.1   26.3*   19.7  22.9*        19.9           24.2
Miami                20.2    19.7  20.0c   19.9        20.0           22.7
Silverstone          20.3   25.3*  26.5*   20.8        20.6           28.9
Monte Carlo         24.6*    20.6   20.7   21.9        20.7           24.3
Jeddah               20.2    21.5   20.8               20.8           21.1
Melbourne           20.9c    20.6  25.7*   23.0        20.9           18.6
Spielberg            21.6    20.7   21.0   21.7        21.3           21.6
Hungaroring          21.4    20.5   21.1   22.3        21.3           21.8
Austin               21.0    21.5   21.3   21.6        21.4           24.2
Baku                 22.3